# Quota dry run (dựng ngạch theo độ dài thật + kiểm tỉ lệ, KHÔNG train)

CPU, phiên tắt Internet. Không chứa test.

In [ ]:
# ===== Cấu hình =====
CAU_HINH = {'che_do': 'quota_dry', 'viec': [{'tap': 'cum', 'config': 'natural', 'seed': 42}, {'tap': 'pair', 'config': 'natural', 'seed': 42}], 'so_eval': 1200, 'lr': 0.0002, 'rank': 16, 'micro_bs': 32, 'tich_luy': 1, 'ngan_sach_token': 4096, 'max_len': 2048, 'seed': 42, 'flt': 0, 'ngan_sach_compute': 4000000, 'so_token_moi_buoc': 8000, 'mo_hinh': {'ten': 'ecomgpt-7b', 'ten_thu_muc': 'EcomGPT-7B', 'khuon': 'ecomgpt', 'attn': 'eager', 'gc': True, 'tok_lay_mau_thu_muc': 'Qwen3-4B'}}

In [ ]:
# Ghi mã vào working (phiên tắt Internet)
from pathlib import Path
Path('cham_diem.py').write_text('"""B4 bộ chấm tự động cho dịch Trung-Việt thương mại điện tử (title + danh sách thuộc tính).\n\nChỉ cần thư viện chuẩn + numpy/pandas/sacrebleu (chạy được trên Kaggle tắt internet); COMET/CometKiwi là tùy chọn.\nChạy:\n  python -I -X utf8 scripts/ft_train/cham_diem.py --tham-chieu REF.jsonl --du-doan HYP.jsonl --ra-thu-muc DIR\n         [--nhanh cum|pair] [--mau-huan-luyen A.jsonl B.jsonl --fold-kiem k] [--comet CKPT] [--kiwi CKPT]\nTừ notebook:  tung_muc_df, tong_hop = cham(tham_chieu_df, du_doan_df, ...)\n\nTham chiếu: id, huong (\'zh->vi\'|\'vi->zh\'), loai (\'title\'|\'mo_ta\'), nhanh (\'cum\'|\'pair\'), category, nguon, dich [, fold5].\nDự đoán: id, hyp [, hyp_tho]. Mục không có hyp tính là hyp rỗng và đếm riêng (thieu_hyp).\n\nCác kiểm riêng đều là HEURISTIC, giới hạn ghi ở docstring từng hàm và ở GIOI_HAN cuối tong_hop.json.\nKhông đụng đến data/processed/ft_run/test_khoa/: đường dẫn do người gọi truyền vào."""\nimport argparse\nimport functools\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sacrebleu\nfrom sacrebleu.metrics import CHRF\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\ntry:\n    from chung import TACH_DONG\nexcept Exception:                                    # Kaggle không có chung.py: dùng bản sao\n    TACH_DONG = {"cum": "; ", "pair": "\\n"}\n\nPHIEN_BAN = "1"\nHUONG_HOP_LE = ("zh->vi", "vi->zh")\n# ngưỡng mặc định; sẽ chốt sau khi xem đầu ra zero-shot (ghi vào tong_hop.json mỗi lần chấm)\nMIN_DOAN, MIN_LAN, HE_SO_DAI, NGUONG_LATIN = 8, 4, 3.0, 0.3\nGIOI_HAN = [\n    "giu_so: số chữ Hán (三, 二十) và số viết chữ tiếng Việt không được nhận; dạng \'1.500\' mơ hồ (1500 hay 1,5) coi là 1500; "\n    "\'25款\' <-> \'2025\' được bỏ qua bằng quy tắc hai chữ số <-> 20xx; quy đổi đơn vị (斤 -> kg) làm số lệch vẫn bị báo",\n    "giu_latin: chỉ là đại diện cho thương hiệu/mã model; nguồn Việt chỉ lấy token có chữ số, có -._/+, có chữ hoa giữa từ, hoặc "\n    "từ viết hoa đầu không phải âm tiết tiếng Việt không dấu (loại nhãn trước \':\' ở mô tả); còn dương tính giả với từ mượn "\n    "(Cotton, Polyester...) dù đã có danh sách loại nhỏ",\n    "sai_chu_viet: đích Trung chứa chữ Việt có dấu bị cờ kể cả khi đó là tên riêng giữ nguyên; đích Việt chứa chữ Hán bị cờ kể cả tên hãng Trung",\n    "dung_so_dong: tách thuần theo TACH_DONG, \'; \' nằm trong giá trị tự do cũng bị tách",\n    "sinh_lap: ngưỡng chưa chốt, đoạn lặp chu kỳ ngắn cần chạy dài >= min_doan*min_lan ký tự mới bị bắt",\n    "cat_loi_dan: một lượt, chỉ các quy tắc ghi rõ; think chưa đóng thẻ thì không cắt; dong_ngu_canh chỉ bắt vài dạng dịch đã biết của lời dẫn \'Product title (context only...)\', "\n    "cách diễn đạt khác thì không bắt và vẫn tính là lỗi của bản dịch",\n    "ref_*: giu_so, giu_latin, sai_chu_viet cũng được chạy trên chính bản tham chiếu (bản dịch máy) để có mốc; chỉ so điểm của mô hình với mốc này, không đọc 100% là đúng/sai tuyệt đối",\n]\n\n_FULL = {ord(chr(0xFF10 + i)): str(i) for i in range(10)}\n_FULL[ord("．")] = "."\n\n\n# ---------------------------------------------------------------- tiện ích\ndef chuan_chuoi(x):\n    """None/NaN -> \'\'; còn lại ép str."""\n    if x is None or (isinstance(x, float) and np.isnan(x)):\n        return ""\n    return str(x)\n\n\ndef tach(text, nhanh):\n    """Tách danh sách thuộc tính theo TACH_DONG[nhanh], bỏ đoạn rỗng."""\n    return [p.strip() for p in chuan_chuoi(text).split(TACH_DONG[nhanh]) if p.strip()]\n\n\ndef chuan_dong(s):\n    """Khóa so khớp dòng: gộp khoảng trắng."""\n    return " ".join(chuan_chuoi(s).split())\n\n\n# ---------------------------------------------------------------- chrF\ndef tao_chrf(huong):\n    """Đích Việt: chrF++ (word_order=2); đích Trung: chrF ký tự (word_order=0)."""\n    if huong not in HUONG_HOP_LE:\n        raise ValueError(f"huong_la:{huong}")\n    return CHRF(word_order=2 if huong.endswith("vi") else 0)\n\n\ndef chrf_cau(huong, hyp, ref):\n    return float(tao_chrf(huong).sentence_score(hyp, [ref]).score)\n\n\ndef chrf_corpus(huong, hyps, refs):\n    if len(hyps) == 0:\n        return None\n    return float(tao_chrf(huong).corpus_score(list(hyps), [list(refs)]).score)\n\n\n# ---------------------------------------------------------------- COMET (tùy chọn)\ndef _tai_comet(duong_dan):\n    if not duong_dan:\n        return None\n    try:\n        from comet import load_from_checkpoint\n    except Exception as e:\n        print(f"[comet] không có thư viện comet, bỏ qua: {e}")\n        return None\n    try:\n        return load_from_checkpoint(duong_dan)\n    except Exception as e:\n        print(f"[comet] không nạp được mô hình {duong_dan}: {e}")\n        return None\n\n\ndef _du_doan_comet(mo_hinh, data, batch):\n    try:\n        import torch\n        gpus = 1 if torch.cuda.is_available() else 0\n    except Exception:\n        gpus = 0\n    try:\n        out = mo_hinh.predict(data, batch_size=batch, gpus=gpus, progress_bar=False)\n        return [float(x) for x in out.scores]\n    except Exception as e:\n        print(f"[comet] lỗi khi chấm: {e}")\n        return None\n\n\ndef cham_comet(nguon, hyp, ref, duong_dan_mo_hinh, batch=16):\n    """COMET có tham chiếu. Trả danh sách điểm theo câu, hoặc None nếu thiếu thư viện/mô hình/lỗi."""\n    mo_hinh = _tai_comet(duong_dan_mo_hinh)\n    if mo_hinh is None:\n        return None\n    return _du_doan_comet(mo_hinh, [{"src": s, "mt": h, "ref": r} for s, h, r in zip(nguon, hyp, ref)], batch)\n\n\ndef cham_kiwi(nguon, hyp, duong_dan_mo_hinh, batch=16):\n    """CometKiwi (không cần tham chiếu). Trả danh sách điểm theo câu, hoặc None như cham_comet."""\n    mo_hinh = _tai_comet(duong_dan_mo_hinh)\n    if mo_hinh is None:\n        return None\n    return _du_doan_comet(mo_hinh, [{"src": s, "mt": h} for s, h in zip(nguon, hyp)], batch)\n\n\n# ---------------------------------------------------------------- giữ số\n_RE_SO = re.compile(r"\\d+(?:[.,]\\d+)*")\n\n\ndef _chuan_so(tok):\n    """Một cụm số thô -> danh sách số chuẩn (chuỗi). Dấu phẩy thập phân -> chấm; bỏ dấu phân nghìn khi nhóm đúng 3 chữ số."""\n    seps = re.findall(r"[.,]", tok)\n    parts = re.split(r"[.,]", tok)\n    khong0 = lambda s: s.lstrip("0") or "0"            # noqa: E731\n    if not seps:\n        return [khong0(tok)]\n    if len(set(seps)) == 2:                            # \'1.234,56\' / \'1,234.56\': dấu cuối là thập phân\n        ngan = seps[-1]\n        phan = tok.replace("." if ngan == "," else ",", "")\n        nguyen, le = phan.split(ngan)\n        le = le.rstrip("0")\n        return [khong0(nguyen) + ("." + le if le else "")]\n    if len(seps) == 1:\n        nguyen, le = parts\n        if len(le) == 3 and nguyen != "0" and len(nguyen) <= 3:        # 2,500 / 1.250 -> hàng nghìn\n            return [khong0(nguyen + le)]\n        le = le.rstrip("0")\n        return [khong0(nguyen) + ("." + le if le else "")]\n    if all(len(p) == 3 for p in parts[1:]) and len(parts[0]) <= 3:      # 1.000.000\n        return [khong0("".join(parts))]\n    return [khong0(p) for p in parts]                  # \'1,2,3\' hay số phiên bản: tách rời\n\n\ndef so_tu_text(text):\n    """Tập số chuẩn hóa trong chuỗi (chữ số toàn chiều -> ASCII)."""\n    t = chuan_chuoi(text).translate(_FULL)\n    kq = set()\n    for m in _RE_SO.findall(t):\n        kq.update(_chuan_so(m))\n    return kq\n\n\ndef giu_so(nguon, hyp):\n    """Số trong nguồn phải có trong hyp và ngược lại. Trả dict(dat, mat, la).\n    Giới hạn: không nhận số chữ (三, \'hai mươi\'); \'1.500\' coi là 1500; \'25款\' ~ \'2025\' được bỏ qua bằng quy tắc 2 chữ số <-> 20xx."""\n    a, b = so_tu_text(nguon), so_tu_text(hyp)\n    mat, la = a - b, b - a\n    for m in list(mat):\n        if m.isdigit() and len(m) == 2 and "20" + m in la:\n            mat.discard(m)\n            la.discard("20" + m)\n    for x in list(la):\n        if x.isdigit() and len(x) == 2 and "20" + x in mat:\n            la.discard(x)\n            mat.discard("20" + x)\n    key = lambda s: (len(s), s)                        # noqa: E731\n    return {"dat": not mat and not la, "mat": sorted(mat, key=key), "la": sorted(la, key=key)}\n\n\n# ---------------------------------------------------------------- giữ token Latin\n_NL = "À-ɏḀ-ỿ"                     # chữ Latin có dấu: token ASCII không được dính vào chúng\n_RE_LATIN = re.compile(rf"(?<![A-Za-z0-9{_NL}])[A-Za-z0-9][A-Za-z0-9\\-\\._/+]{{1,}}(?![A-Za-z0-9{_NL}])")\n_DON_VI = ("kg|g|mg|ml|l|cm|mm|m|km|w|v|a|hz|khz|mhz|ghz|mah|gb|tb|mb|kb|oz|lb|lbs|pcs|pc|cl|dl|ah|kw|kv|ma|mw|db|cc|gr")\n_RE_DON_VI = re.compile(rf"^[\\d.,x*×/\\-+]*(?:{_DON_VI})$")\n# âm tiết tiếng Việt không dấu: phụ âm đầu + 1-3 nguyên âm + phụ âm cuối\n_RE_AM_TIET = re.compile(r"^(?:ngh|ng|nh|ph|th|tr|ch|kh|gh|gi|qu|[bcdghklmnpqrstvx])?[aeiouy]{1,3}(?:ng|nh|ch|[cmnpt])?$")\n_TU_MUON = {"cotton", "polyester", "nylon", "spandex", "inox", "silicone", "plastic", "metal", "leather", "latex", "acrylic",\n            "combo", "freeship", "size"}\n\n\ndef _bo_nhan(nguon, nhanh):\n    """Mô tả: bỏ phần nhãn trước dấu \':\' đầu tiên của mỗi thuộc tính, chỉ giữ giá trị."""\n    kq = []\n    for dong in chuan_chuoi(nguon).split(TACH_DONG[nhanh]):\n        i = min([k for k in (dong.find(":"), dong.find("：")) if k >= 0], default=-1)\n        kq.append(dong[i + 1:] if i >= 0 else dong)\n    return " ".join(kq)\n\n\ndef _nhan_vi(t):\n    """Nguồn Việt: token ASCII nào đáng coi là tên riêng/mã (heuristic, xem GIOI_HAN)."""\n    if any(c.isdigit() for c in t) or re.search(r"[\\-\\._/+]", t):\n        return True\n    if any(c.isupper() for c in t[1:]):                # ALLCAPS hay iPhone\n        return True\n    return t[0].isupper() and len(t) >= 3 and not _RE_AM_TIET.match(t.lower()) and t.lower() not in _TU_MUON\n\n\ndef token_latin(nguon, huong="zh->vi", loai="title", nhanh="cum"):\n    """Token Latin/số-chữ đại diện thương hiệu/mã model, chữ thường, không trùng lặp, theo thứ tự xuất hiện."""\n    text = chuan_chuoi(nguon).translate(_FULL)\n    viet = huong.startswith("vi")\n    if viet and loai == "mo_ta":\n        text = _bo_nhan(text, nhanh)\n    kq = []\n    for m in _RE_LATIN.finditer(text):\n        t = m.group(0).rstrip("-._/+")\n        if len(t) < 2 or not any(c.isalpha() for c in t) or _RE_DON_VI.match(t.lower()):\n            continue\n        if viet and not _nhan_vi(t):\n            continue\n        if t.lower() not in kq:\n            kq.append(t.lower())\n    return kq\n\n\ndef _gon(s):\n    return re.sub(r"[\\s\\-\\._/+]", "", s)\n\n\ndef giu_latin(nguon, hyp, huong="zh->vi", loai="title", nhanh="cum"):\n    """Mọi token Latin của nguồn phải có trong hyp (không phân biệt hoa thường, so cả dạng bỏ khoảng trắng/gạch). Dict(dat, mat, tong).\n    Heuristic: thương hiệu/mã model; dương tính giả với từ mượn được dịch hẳn sang chữ khác."""\n    toks = token_latin(nguon, huong, loai, nhanh)\n    h = chuan_chuoi(hyp).translate(_FULL).lower()\n    hg = _gon(h)\n    mat = [t for t in toks if t not in h and _gon(t) not in hg]\n    return {"dat": not mat, "mat": mat, "tong": len(toks)}\n\n\n# ---------------------------------------------------------------- sai chữ viết\n_RE_HAN = re.compile(r"[㐀-䶿一-鿿豈-\ufaff]")\n_VIET_DAU = set("àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ")\n_VIET_DAU |= {c.upper() for c in _VIET_DAU}\n\n\ndef sai_chu_viet(huong, nguon, hyp, nguong_latin=NGUONG_LATIN):\n    """Đích Việt chứa chữ Hán -> cờ. Đích Trung chứa chữ Việt có dấu, hoặc tỉ lệ chữ Latin (không tính từ ASCII đã có trong nguồn)\n    trên tổng ký tự chữ > nguong_latin (và >= 4 chữ) -> cờ. Dict(co, ly_do, so_han, so_viet_dau, ti_le_latin)."""\n    h = chuan_chuoi(hyp)\n    so_han = len(_RE_HAN.findall(h))\n    if huong.endswith("vi"):\n        return {"co": so_han > 0, "ly_do": "co_chu_han" if so_han else "", "so_han": so_han, "so_viet_dau": 0, "ti_le_latin": 0.0}\n    so_dau = sum(1 for c in h if c in _VIET_DAU)\n    trong_nguon = {r.lower() for r in re.findall(r"[A-Za-z]{2,}", chuan_chuoi(nguon))}\n    con_lai = sum(len(r) for r in re.findall(r"[A-Za-z]{2,}", h) if r.lower() not in trong_nguon)\n    tong = sum(1 for c in h if c.isalpha())\n    ti_le = con_lai / tong if tong else 0.0\n    ly = []\n    if so_dau:\n        ly.append("chu_viet_dau")\n    if con_lai >= 4 and ti_le > nguong_latin:\n        ly.append("nhieu_latin")\n    return {"co": bool(ly), "ly_do": "+".join(ly), "so_han": so_han, "so_viet_dau": so_dau, "ti_le_latin": ti_le}\n\n\n# ---------------------------------------------------------------- số dòng, sinh lặp, độ dài\ndef dung_so_dong(hyp, ref, nhanh):\n    """Số thuộc tính hyp == số thuộc tính tham chiếu (tách theo TACH_DONG[nhanh]). Chỉ dùng cho loai == \'mo_ta\'."""\n    return len(tach(hyp, nhanh)) == len(tach(ref, nhanh))\n\n\n@functools.lru_cache(maxsize=None)\ndef _re_lap(min_doan, min_lan):\n    return re.compile(rf"(.{{{min_doan},{max(200, min_doan)}}}?)\\1{{{min_lan - 1},}}")\n\n\ndef sinh_lap(hyp, ref, min_doan=MIN_DOAN, min_lan=MIN_LAN, he_so_dai=HE_SO_DAI):\n    """Lặp vô hạn: đoạn >= min_doan ký tự (bỏ khoảng trắng) lặp liên tiếp >= min_lan lần, hoặc len(hyp) > he_so_dai*len(ref)+20.\n    Dict(co, ly_do)."""\n    h = chuan_chuoi(hyp)\n    s = re.sub(r"\\s+", "", h)\n    ly = []\n    if len(s) >= min_doan * min_lan and _re_lap(min_doan, min_lan).search(s):\n        ly.append("doan_lap")\n    if len(h) > he_so_dai * len(chuan_chuoi(ref)) + 20:\n        ly.append("qua_dai")\n    return {"co": bool(ly), "ly_do": "+".join(ly)}\n\n\ndef ti_le_do_dai(hyp, ref):\n    n = len(chuan_chuoi(ref))\n    return len(chuan_chuoi(hyp)) / n if n else float("nan")\n\n\n# ---------------------------------------------------------------- dòng đã thấy / mới\ndef xay_tap_da_thay(mau, fold_kiem=None, nhanh=None):\n    """Tập (category, dòng_nguồn) từ mẫu HUẤN LUYỆN (DataFrame cột loai, category, nguon, nhanh, fold5).\n    Có fold5: chỉ lấy fold5 >= 0, và nếu fold_kiem != None thì bỏ fold5 == fold_kiem. Chỉ loai == \'mo_ta\'."""\n    d = mau[mau.loai == "mo_ta"]\n    if "fold5" in d.columns:\n        d = d[d.fold5 >= 0]\n        if fold_kiem is not None:\n            d = d[d.fold5 != fold_kiem]\n    tap = set()\n    for r in d.itertuples(index=False):\n        nh = getattr(r, "nhanh", None) or nhanh\n        for dong in tach(r.nguon, nh):\n            tap.add((chuan_chuoi(getattr(r, "category", "")), chuan_dong(dong)))\n    return tap\n\n\n# ---------------------------------------------------------------- cắt lời dẫn\n_INTRO = (r"(?:(?:here\\s+is|here\'s|here\\s+are|below\\s+is|this\\s+is)\\s+(?:the\\s+)?(?:(?:vietnamese|chinese|english|final)\\s+)?"\n          r"(?:translation|translated\\s+text)(?:\\s+of\\s+[^:：\\n]{1,60})?"\n          r"|(?:vietnamese\\s+|chinese\\s+)?translation|translated\\s+text"\n          r"|(?:(?:đây\\s+là|dưới\\s+đây\\s+là)\\s+)?bản\\s+dịch(?:\\s+tiếng\\s+(?:việt|trung))?|dịch"\n          r"|(?:以下是)?翻[译譯](?:结果)?|译文)")\n_RE_DONG_DAN = re.compile(rf"^\\s*{_INTRO}\\s*[:：]?\\s*$", re.I)\n_RE_TIEN_TO = re.compile(rf"^\\s*{_INTRO}\\s*[:：]\\s*(\\S.*)$", re.I | re.S)\n# Dòng chép lại lời dẫn "Product title (context only, do not translate): ..." đã bị dịch (zero-shot hay chép; Nhật Anh duyệt sửa 07/11/2026 trước khi đo test).\n# CHỈ bắt các dạng đã biết: 产品标题（仅上下文…, Tiêu đề/Tên sản phẩm (chỉ…, Product title (context only. Dạng diễn đạt khác không bắt (ghi ở GIOI_HAN).\n_RE_NGU_CANH = re.compile(r"^\\s*(?:产品(?:标题|名称)\\s*[（(]\\s*仅|(?:Tiêu đề|Tên)\\s+sản phẩm\\s*[（(]\\s*chỉ|Product title\\s*[（(]\\s*context only)", re.I)\n_RE_THINK = re.compile(r"<think>.*?</think>", re.S | re.I)\n_RE_RAO = re.compile(r"^\\s*```[^\\n`]*\\n(.*?)\\n?[ \\t]*```\\s*$", re.S)\n_RE_RAO_DONG = re.compile(r"^\\s*```(.*?)```\\s*$", re.S)\n_CAP_NGOAC = {\'"\': \'"\', "“": "”", "‘": "’", "\'": "\'", "«": "»", "「": "」", "『": "』", "„": "“"}\n\n\ndef cat_loi_dan(text):\n    """Cắt lời dẫn, một lượt, theo thứ tự (tên quy tắc ghi trong chuỗi trả về, ngăn bằng \',\'):\n      think        bỏ mọi khối <think>...</think> (kể cả rỗng);\n      rao_code     bỏ rào ``` bao toàn bộ (kể cả dòng ```lang);\n      dong_ngu_canh bỏ MỌI dòng chép lại lời dẫn "Product title (context only, ...)" đã dịch (\'产品标题（仅上下文…\', \'Tiêu đề sản phẩm (chỉ…\'); chỉ các dạng này;\n      dong_dan     bỏ dòng đầu (không rỗng) nếu CẢ DÒNG là lời dẫn: \'Here is the translation:\', \'Translation:\', \'Bản dịch:\', \'Dịch:\',\n                   \'翻译：\', \'译文：\' (không phân biệt hoa thường, có/không dấu hai chấm);\n      tien_to_dan  nếu lời dẫn nằm cùng dòng với nội dung thì phải CÓ dấu hai chấm (\'Translation: Áo thun\' -> \'Áo thun\');\n                   \'Dịch vụ bảo hành: Có\' giữ nguyên vì sau \'Dịch\' không phải dấu hai chấm;\n      ngoac_kep    bỏ cặp ngoặc kép bao toàn bộ nếu bên trong không còn ngoặc cùng loại;\n      rồi strip hai đầu (không tính là \'đã cắt\').\n    Trả (sạch, đã_cắt, quy_tắc)."""\n    t = chuan_chuoi(text)\n    qt = []\n    t2 = _RE_THINK.sub("", t)\n    if t2 != t:\n        qt.append("think")\n        t = t2\n    t = t.strip()\n    m = _RE_RAO.match(t) or _RE_RAO_DONG.match(t)\n    if m:\n        t = m.group(1).strip()\n        qt.append("rao_code")\n    dong = t.split("\\n")\n    con = [d for d in dong if not _RE_NGU_CANH.match(d)]\n    if len(con) != len(dong):\n        t = "\\n".join(con).strip()\n        dong = t.split("\\n")\n        qt.append("dong_ngu_canh")\n    i = next((k for k, d in enumerate(dong) if d.strip()), None)\n    if i is not None:\n        if _RE_DONG_DAN.match(dong[i]):\n            t = "\\n".join(dong[i + 1:]).strip()\n            qt.append("dong_dan")\n        else:\n            m = _RE_TIEN_TO.match(dong[i])\n            if m:\n                t = "\\n".join([m.group(1)] + dong[i + 1:]).strip()\n                qt.append("tien_to_dan")\n    if len(t) >= 2 and t[0] in _CAP_NGOAC and t[-1] == _CAP_NGOAC[t[0]]:\n        trong = t[1:-1]\n        if t[0] not in trong and _CAP_NGOAC[t[0]] not in trong:\n            t = trong\n            qt.append("ngoac_kep")\n    return t.strip(), bool(qt), ",".join(qt)\n\n\n# ---------------------------------------------------------------- chấm\ndef _chuan_tham_chieu(tc, nhanh):\n    d = tc.copy()\n    for c in ("id", "huong", "loai", "nguon", "dich"):\n        if c not in d.columns:\n            raise ValueError(f"thieu_cot_tham_chieu:{c}")\n    d["id"] = d["id"].astype(str)\n    if d["id"].duplicated().any():\n        raise ValueError("id_tham_chieu_trung")\n    if "category" not in d.columns:\n        d["category"] = ""\n    d["category"] = d["category"].map(chuan_chuoi)\n    if "nhanh" in d.columns:\n        if nhanh:\n            d = d[d["nhanh"] == nhanh]\n    else:\n        if nhanh is None and (d["loai"] == "mo_ta").any():\n            raise ValueError("thieu_nhanh: truyền nhanh hoặc thêm cột nhanh")\n        d["nhanh"] = nhanh or ""\n    bad = set(d["huong"]) - set(HUONG_HOP_LE)\n    if bad:\n        raise ValueError(f"huong_la:{bad}")\n    return d.reset_index(drop=True)\n\n\ndef _lay_du_doan(dd):\n    """id -> hyp; trả (dict, số id trùng). hyp thiếu/NaN không vào dict."""\n    kq, trung = {}, 0\n    for r in dd.to_dict("records"):\n        k = str(r["id"])\n        if k in kq:\n            trung += 1\n        h = r.get("hyp")\n        if h is None or (isinstance(h, float) and np.isnan(h)):\n            kq.pop(k, None)\n            continue\n        kq[k] = str(h)\n    return kq, trung\n\n\ndef cham(tham_chieu, du_doan, nhanh=None, tap_da_thay=None, comet=None, kiwi=None, batch=16, min_doan=MIN_DOAN,\n         min_lan=MIN_LAN, he_so_dai=HE_SO_DAI, nguong_latin=NGUONG_LATIN, cat_loi_dan_bat=True):\n    """Chấm toàn bộ. Trả (tung_muc_df, tong_hop_dict).\n    tap_da_thay: tập (category, dòng_nguồn) từ xay_tap_da_thay(); None thì bỏ phần dòng đã thấy/mới.\n    comet / kiwi: đường dẫn checkpoint (tùy chọn). Cột bắt đầu bằng \'_\' là phụ trợ, không ghi ra file."""\n    ref = _chuan_tham_chieu(tham_chieu, nhanh)\n    hyps, n_trung = _lay_du_doan(du_doan)\n    ids_ref = set(ref["id"])\n    n_thua = len(set(hyps) - ids_ref)\n    rows = []\n    for r in ref.to_dict("records"):\n        huong, loai, nh = r["huong"], r["loai"], r["nhanh"]\n        nguon, dich = chuan_chuoi(r["nguon"]), chuan_chuoi(r["dich"])\n        thieu = r["id"] not in hyps\n        hyp = hyps.get(r["id"], "")\n        if cat_loi_dan_bat:\n            hyp, da_cat, qt = cat_loi_dan(hyp)\n        else:\n            hyp, da_cat, qt = hyp, False, ""\n        so, la = giu_so(nguon, hyp), giu_latin(nguon, hyp, huong, loai, nh)\n        sc, lap = sai_chu_viet(huong, nguon, hyp, nguong_latin), sinh_lap(hyp, dich, min_doan, min_lan, he_so_dai)\n        rso, rla, rsc = giu_so(nguon, dich), giu_latin(nguon, dich, huong, loai, nh), sai_chu_viet(huong, nguon, dich, nguong_latin)      # mốc: cùng kiểm trên bản tham chiếu\n        row = {"ref_giu_so_dat": rso["dat"], "ref_giu_latin_dat": rla["dat"], "ref_sai_chu_viet": rsc["co"],"id": r["id"], "huong": huong, "loai": loai, "category": r["category"], "nhanh": nh, "thieu_hyp": thieu,\n               "hyp_cham": hyp, "da_cat_loi_dan": da_cat, "quy_tac_cat": qt, "chrf": chrf_cau(huong, hyp, dich),\n               "giu_so_dat": so["dat"], "so_mat": so["mat"], "so_la": so["la"],\n               "giu_latin_dat": la["dat"], "latin_mat": la["mat"], "latin_tong": la["tong"],\n               "sai_chu_viet": sc["co"], "sai_chu_viet_ly_do": sc["ly_do"], "so_han": sc["so_han"],\n               "so_viet_dau": sc["so_viet_dau"], "ti_le_latin": sc["ti_le_latin"],\n               "dung_so_dong": None, "so_dong_hyp": None, "so_dong_ref": None,\n               "sinh_lap": lap["co"], "sinh_lap_ly_do": lap["ly_do"], "ti_le_do_dai": ti_le_do_dai(hyp, dich),\n               "_nguon": nguon, "_ref": dich}\n        if loai == "mo_ta":\n            hl, rl, sl = tach(hyp, nh), tach(dich, nh), tach(nguon, nh)\n            row.update(dung_so_dong=len(hl) == len(rl), so_dong_hyp=len(hl), so_dong_ref=len(rl))\n            if tap_da_thay is not None:\n                if len(hl) == len(rl) == len(sl):\n                    thay = [(r["category"], chuan_dong(x)) in tap_da_thay for x in sl]\n                    row.update(can_dong=True, n_dong_da_thay=sum(thay), n_dong_moi=len(thay) - sum(thay),\n                               _dong=list(zip(thay, hl, rl)))\n                else:\n                    row.update(can_dong=False, n_dong_da_thay=None, n_dong_moi=None, _dong=None)\n        rows.append(row)\n    df = pd.DataFrame(rows)\n    ghi_chu_comet = {}\n    for ten, duong_dan in (("comet", comet), ("kiwi", kiwi)):\n        if not duong_dan:\n            continue\n        if ten == "comet":\n            diem = cham_comet(df["_nguon"].tolist(), df["hyp_cham"].tolist(), df["_ref"].tolist(), duong_dan, batch)\n        else:\n            diem = cham_kiwi(df["_nguon"].tolist(), df["hyp_cham"].tolist(), duong_dan, batch)\n        ghi_chu_comet[ten] = "da_chay" if diem is not None else "khong_chay_duoc (thieu thu vien comet hoac mo hinh)"\n        if diem is not None:\n            df[ten] = diem\n    cau_hinh = {"min_doan": min_doan, "min_lan": min_lan, "he_so_dai": he_so_dai, "nguong_latin": nguong_latin,\n                "cat_loi_dan": cat_loi_dan_bat, "nhanh": nhanh, "co_dong_da_thay_moi": tap_da_thay is not None,\n                "so_dong_da_thay_trong_tap": None if tap_da_thay is None else len(tap_da_thay)}\n    return df, tong_hop(df, cau_hinh, n_thua, n_trung, ghi_chu_comet)\n\n\n# ---------------------------------------------------------------- tổng hợp\ndef _tom_tat(d, huong, day_du=True):\n    out = {"n": int(len(d)), "thieu_hyp": int(d["thieu_hyp"].sum()),\n           "chrf_corpus": chrf_corpus(huong, d["hyp_cham"], d["_ref"]), "chrf_cau_tb": float(d["chrf"].mean())}\n    if not day_du:\n        return out\n    out.update(\n        ti_le_giu_so_dat=float(d["giu_so_dat"].mean()), ti_le_giu_latin_dat=float(d["giu_latin_dat"].mean()),\n        ti_le_sai_chu_viet=float(d["sai_chu_viet"].mean()), ti_le_sinh_lap=float(d["sinh_lap"].mean()),\n        ti_le_da_cat_loi_dan=float(d["da_cat_loi_dan"].mean()),\n        ti_le_do_dai_tb=float(d["ti_le_do_dai"].mean()), ti_le_do_dai_trung_vi=float(d["ti_le_do_dai"].median()))\n    for c, ten in (("ref_giu_so_dat", "ref_ti_le_giu_so_dat"), ("ref_giu_latin_dat", "ref_ti_le_giu_latin_dat"), ("ref_sai_chu_viet", "ref_ti_le_sai_chu_viet")):\n        if c in d.columns:\n            out[ten] = float(d[c].mean())\n    if d["dung_so_dong"].notna().any():\n        out["ti_le_dung_so_dong"] = float(d["dung_so_dong"].dropna().astype(bool).mean())\n    for c in ("comet", "kiwi"):\n        if c in d.columns:\n            out[f"{c}_tb"] = float(d[c].mean())\n    return out\n\n\ndef tong_hop(df, cau_hinh, n_hyp_thua=0, n_trung=0, ghi_chu_comet=None):\n    """Nhóm (huong, loai), (huong, loai, category); điểm chính = macro theo hướng (không trọng số theo số mục/token)."""\n    nhom = {}\n    for (h, l), d in df.groupby(["huong", "loai"], sort=True):\n        nhom[f"{h}/{l}"] = _tom_tat(d, h)\n    theo_cat = {}\n    for (h, l, c), d in df.groupby(["huong", "loai", "category"], sort=True):\n        theo_cat[f"{h}/{l}/{c}"] = _tom_tat(d, h, day_du=False)\n    theo_huong = {}\n    for h in sorted(df["huong"].unique()):\n        loai_co = [l for l in ("title", "mo_ta") if f"{h}/{l}" in nhom]\n        theo_huong[h] = {"loai_gom": loai_co,\n                         "chrf_corpus": float(np.mean([nhom[f"{h}/{l}"]["chrf_corpus"] for l in loai_co])),\n                         "chrf_cau_tb": float(np.mean([nhom[f"{h}/{l}"]["chrf_cau_tb"] for l in loai_co]))}\n    diem_chinh = {"cach_tinh": "chrF corpus: mỗi hướng = trung bình đều của title và mo_ta; điểm chính = trung bình đều các hướng "\n                               "(zh->vi dùng chrF++, vi->zh dùng chrF ký tự; không trọng số theo số mục/token)",\n                  "huong_gom": sorted(theo_huong),\n                  "chrf_macro_huong": float(np.mean([v["chrf_corpus"] for v in theo_huong.values()])),\n                  "chrf_cau_tb_macro_huong": float(np.mean([v["chrf_cau_tb"] for v in theo_huong.values()]))}\n    cat = {}\n    for (h, l), d in df.groupby(["huong", "loai"], sort=True):\n        cat[f"{h}/{l}"] = {"so_cat": int(d["da_cat_loi_dan"].sum()), "ti_le": float(d["da_cat_loi_dan"].mean()),\n                           "theo_quy_tac": {k: int(v) for k, v in\n                                            pd.Series([x for q in d["quy_tac_cat"] for x in q.split(",") if x]).value_counts().items()}}\n    tm = {"phien_ban": {"cham_diem": PHIEN_BAN, "sacrebleu": sacrebleu.__version__},\n          "cau_hinh": cau_hinh, "so_muc": int(len(df)), "thieu_hyp": int(df["thieu_hyp"].sum()),\n          "thieu_hyp_theo_nhom": {k: v["thieu_hyp"] for k, v in nhom.items()},\n          "hyp_thua_khong_co_trong_tham_chieu": int(n_hyp_thua), "id_du_doan_trung": int(n_trung),\n          "diem_chinh": diem_chinh, "theo_huong": theo_huong, "theo_nhom": nhom, "theo_category": theo_cat,\n          "cat_loi_dan": cat, "comet": ghi_chu_comet or {}, "gioi_han": GIOI_HAN}\n    if "can_dong" in df.columns:\n        dm = {}\n        for h, d in df[df["loai"] == "mo_ta"].groupby("huong", sort=True):\n            can = d[d["can_dong"] == True]       # noqa: E712\n            dong = [x for l in can["_dong"] for x in l]\n            ph = {}\n            for ten, co in (("da_thay", True), ("moi", False)):\n                p = [(hy, rf) for t, hy, rf in dong if t == co]\n                ph[ten] = {"n_dong": len(p), "chrf_corpus": chrf_corpus(h, [a for a, _ in p], [b for _, b in p])}\n            ph["so_muc_khong_can_duoc"] = int(len(d) - len(can))\n            ph["so_muc_mo_ta"] = int(len(d))\n            dm[h] = ph\n        tm["dong_da_thay_moi"] = dm\n    return tm\n\n\n# ---------------------------------------------------------------- vào/ra\ndef _sach(o):\n    if isinstance(o, dict):\n        return {str(k): _sach(v) for k, v in o.items()}\n    if isinstance(o, (list, tuple, set)):\n        return [_sach(v) for v in o]\n    if isinstance(o, (np.bool_,)):\n        return bool(o)\n    if isinstance(o, np.integer):\n        return int(o)\n    if isinstance(o, (float, np.floating)):\n        return None if np.isnan(o) or np.isinf(o) else float(o)\n    return o\n\n\ndef doc_jsonl(path, cot=None):\n    rows = []\n    with open(path, encoding="utf-8") as f:\n        for dong in f:\n            if dong.strip():\n                o = json.loads(dong)\n                rows.append({k: o.get(k) for k in cot} if cot else o)\n    return pd.DataFrame(rows)\n\n\ndef ghi_ket_qua(df, tm, thu_muc):\n    thu_muc = Path(thu_muc)\n    thu_muc.mkdir(parents=True, exist_ok=True)\n    cot = [c for c in df.columns if not c.startswith("_")]\n    with open(thu_muc / "diem_tung_muc.jsonl", "w", encoding="utf-8") as f:\n        for r in df[cot].to_dict("records"):\n            f.write(json.dumps(_sach(r), ensure_ascii=False) + "\\n")\n    (thu_muc / "tong_hop.json").write_text(json.dumps(_sach(tm), ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.split("\\n")[0])\n    ap.add_argument("--tham-chieu", required=True)\n    ap.add_argument("--du-doan", required=True)\n    ap.add_argument("--ra-thu-muc", required=True)\n    ap.add_argument("--nhanh", choices=["cum", "pair"])\n    ap.add_argument("--mau-huan-luyen", nargs="+")\n    ap.add_argument("--fold-kiem", type=int)\n    ap.add_argument("--comet")\n    ap.add_argument("--kiwi")\n    ap.add_argument("--batch", type=int, default=16)\n    ap.add_argument("--min-doan", type=int, default=MIN_DOAN)\n    ap.add_argument("--min-lan", type=int, default=MIN_LAN)\n    ap.add_argument("--he-so-dai", type=float, default=HE_SO_DAI)\n    ap.add_argument("--nguong-latin", type=float, default=NGUONG_LATIN)\n    ap.add_argument("--khong-cat-loi-dan", action="store_true")\n    a = ap.parse_args(argv)\n    tc, dd = doc_jsonl(a.tham_chieu), doc_jsonl(a.du_doan, ["id", "hyp"])\n    tap = None\n    if a.mau_huan_luyen:\n        mau = pd.concat([doc_jsonl(p, ["id", "category", "loai", "nhanh", "fold5", "nguon"]) for p in a.mau_huan_luyen],\n                        ignore_index=True)\n        tap = xay_tap_da_thay(mau, a.fold_kiem, a.nhanh)\n    df, tm = cham(tc, dd, nhanh=a.nhanh, tap_da_thay=tap, comet=a.comet, kiwi=a.kiwi, batch=a.batch, min_doan=a.min_doan,\n                  min_lan=a.min_lan, he_so_dai=a.he_so_dai, nguong_latin=a.nguong_latin, cat_loi_dan_bat=not a.khong_cat_loi_dan)\n    if a.fold_kiem is not None:\n        tm["cau_hinh"]["fold_kiem"] = a.fold_kiem\n    ghi_ket_qua(df, tm, a.ra_thu_muc)\n    dc = tm["diem_chinh"]\n    print(f"chrF macro theo hướng: {dc[\'chrf_macro_huong\']:.2f}  (số mục {tm[\'so_muc\']}, thiếu hyp {tm[\'thieu_hyp\']})")\n    for h, v in tm["theo_huong"].items():\n        print(f"  {h}: {v[\'chrf_corpus\']:.2f}  ({\'+\'.join(v[\'loai_gom\'])})")\n    return df, tm\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
Path('kaggle_train.py').write_text('"""Huấn luyện Qwen3-4B + LoRA trên Kaggle (RTX Pro 6000, phiên tắt Internet). Notebook nlp-group3 nhúng file này và gọi main(CAU_HINH).\n\nche_do="thu_b5" (B5): chạy so_buoc bước để kiểm môi trường: loss giảm, lưu/nạp lại checkpoint cho cùng loss, đo token/giây và bộ nhớ, sinh thử\nvài mẫu của fold kiểm bằng mô hình fine-tune và zero-shot. che_do="b6" (B6): dò loại dần trên 1 fold, xem chay_b6.\nche_do="b78" (B7, B8): cấu hình đã chốt chạy trên pair fold 0 và cụm fold 1, 2, chấm thêm LoRA attention, xem chay_b78.\nKHÓA TEST: chỉ đọc pool_*.jsonl.gz, dừng nếu thấy file test trong /kaggle/input.\nCác hàm thuần (xay_mau, chia_lo, lich_lr, chon_vong2, ...) không cần torch để test được trên máy (test_kaggle_train.py).\n"""\nimport gzip\nimport json\nimport math\nimport random\nimport subprocess\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\n\nVAO = Path("/kaggle/input")\nRA = Path("/kaggle/working")\nNHAC_EOS = "<|im_end|>"\n\n\n# ---------------------------------------------------------------- hàm thuần\ndef xay_mau(mau, tok, max_len):\n    """Mã hóa mỗi mẫu: prompt theo khuôn chat (tắt suy luận), đích kết thúc bằng <|im_end|>. Loss chỉ trên phần dịch (labels -100 cho prompt).\n    Trả (danh sách {id, ids, labels, fold5}, số mẫu vượt max_len bị bỏ)."""\n    ra, bo = [], 0\n    for m in mau:\n        p = tok.apply_chat_template(m["messages"][:1], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        a = m["messages"][1]["content"] + NHAC_EOS + "\\n"\n        ip, ia = tok(p, add_special_tokens=False)["input_ids"], tok(a, add_special_tokens=False)["input_ids"]\n        if len(ip) + len(ia) > max_len:\n            bo += 1\n            continue\n        ra.append({"id": m["id"], "ids": ip + ia, "labels": [-100] * len(ip) + ia, "fold5": m["fold5"],\n                   "huong": m.get("huong"), "loai": m.get("loai"), "category": m.get("category"), "product_id": m.get("product_id")})\n    return ra, bo\n\n\ndef cat_ngan_sach(k, micro_bs, ngan_sach):\n    """k đã xếp theo độ dài: cắt thành lô <= micro_bs mẫu và (số mẫu x mẫu dài nhất) <= ngan_sach token, tránh tràn GPU ở lô dài."""\n    lo, hien = [], []\n    for m in k:\n        if hien and (len(hien) >= micro_bs or (len(hien) + 1) * len(m["ids"]) > ngan_sach):\n            lo.append(hien)\n            hien = []\n        hien.append(m)\n    return lo + [hien] if hien else lo\n\n\ndef chia_lo(mau, micro_bs, seed, ngan_sach=8192, he_so_xo=50):\n    """Lô gần cùng độ dài để ít đệm: xáo, cắt khúc he_so_xo*micro_bs, xếp theo độ dài trong khúc, chia lô theo ngân sách, rồi xáo thứ tự lô."""\n    rng = random.Random(seed)\n    x = list(mau)\n    rng.shuffle(x)\n    lo = []\n    khuc = he_so_xo * micro_bs\n    for i in range(0, len(x), khuc):\n        lo += cat_ngan_sach(sorted(x[i:i + khuc], key=lambda m: len(m["ids"])), micro_bs, ngan_sach)\n    rng.shuffle(lo)\n    return lo\n\n\ndef gom_buoc(lo, co_batch):\n    """Gom micro-batch thành bước optimizer: mỗi bước gom tới khi đủ co_batch mẫu. Đếm trước để lịch lr có đúng tổng số bước (số mẫu mỗi bước có thể hơi vượt)."""\n    buoc, hien, dem = [], [], 0\n    for k in lo:\n        hien.append(k)\n        dem += len(k)\n        if dem >= co_batch:\n            buoc.append(hien)\n            hien, dem = [], 0\n    return buoc + [hien] if hien else buoc\n\n\ndef lich_lr(buoc, tong, ti_le_warmup=0.03):\n    """Hệ số lr: tăng tuyến tính trong warmup rồi giảm theo cosine về 0."""\n    w = max(1, int(tong * ti_le_warmup))\n    if buoc < w:\n        return (buoc + 1) / w\n    return 0.5 * (1 + math.cos(math.pi * (buoc - w) / max(1, tong - w)))\n\n\n# B6: 6 cấu hình của lưới + 1 cấu hình LoRA chỉ attention (cùng lr, rank với lr1e-4_r16, để biết có dùng được kiểu nhanh hơn không)\nCAU_HINH_B6 = [{"ten": f"lr{lr}_r{r}", "lr": float(lr), "rank": r} for r in (16, 64) for lr in ("5e-5", "1e-4", "2e-4")]\nCAU_HINH_B6.append({"ten": "lr1e-4_r16_attn", "lr": 1e-4, "rank": 16, "muc_tieu": ["q_proj", "k_proj", "v_proj", "o_proj"]})\nSO_EPOCH_B6 = 2       # tối đa 2 epoch (05/11/2026), các bước sau cũng dùng 2 epoch\n\n\nclass DungHetGio(Exception):\n    """Dừng có trật tự: đã lưu trạng thái, phiên sau chạy tiếp được."""\n\n\ndef diem_macro(loss_nhom):\n    """loss_nhom {\'huong|loai\': loss}. Mỗi hướng = trung bình đều title và mo_ta; điểm = trung bình đều các hướng (không theo token)."""\n    theo_huong = {}\n    for k, v in loss_nhom.items():\n        theo_huong.setdefault(k.split("|")[0], []).append(v)\n    return sum(sum(x) / len(x) for x in theo_huong.values()) / len(theo_huong)\n\n\ndef chon_vong2(diem, moi=0.01, toi_thieu=2, toi_da=3):\n    """diem {tên: loss, thấp là tốt}. Lấy toi_thieu cấu hình đầu, thêm cấu hình lệch không quá moi so với đầu bảng, trần toi_da.\n    Bỏ cấu hình loss không hữu hạn."""\n    x = sorted((v, k) for k, v in diem.items() if math.isfinite(v))\n    if not x:\n        raise RuntimeError("khong_con_cau_hinh_hop_le")\n    chon = [k for _, k in x[:toi_thieu]]\n    chon += [k for v, k in x[toi_thieu:toi_da] if v <= x[0][0] * (1 + moi)]\n    return chon\n\n\ndef lay_phan_tang(mau, moi_nhom, seed):\n    """Mẫu cố định theo (hướng, loại), dùng chung cho mọi cấu hình để so sánh theo cặp."""\n    rng = random.Random(seed)\n    nhom = {}\n    for m in sorted(mau, key=lambda m: m["id"]):\n        nhom.setdefault(f"{m[\'huong\']}|{m[\'loai\']}", []).append(m)\n    ra = []\n    for k in sorted(nhom):\n        ra += rng.sample(nhom[k], min(moi_nhom, len(nhom[k])))\n    return ra\n\n\ndef bootstrap_chenh(a, b, nhom, so_lan=2000, seed=42):\n    """a, b: điểm từng mục của hai cấu hình trên cùng mục; nhom: \'huong|loai\' từng mục. Khoảng 95% của chênh (a - b) theo điểm macro, lấy mẫu lại trong từng nhóm."""\n    rng = random.Random(seed)\n    theo = {}\n    for x, y, n in zip(a, b, nhom):\n        theo.setdefault(n, []).append(x - y)\n    kq = []\n    for _ in range(so_lan):\n        d = {n: sum(rng.choices(v, k=len(v))) / len(v) for n, v in theo.items()}\n        kq.append(diem_macro(d))\n    kq.sort()\n    return {"chenh": diem_macro({n: sum(v) / len(v) for n, v in theo.items()}), "thap": kq[int(0.025 * so_lan)], "cao": kq[int(0.975 * so_lan) - 1]}\n\n\n# B7, B8 (Nhật Anh duyệt 05/11/2026): cấu hình chốt từ B6; (tên việc, nhánh, fold kiểm). cum_f0 không huấn luyện, dùng lại adapter B6\nCAU_HINH_CHOT = {"ten": "lr2e-4_r64", "lr": 2e-4, "rank": 64}\nVIEC_B78 = [("cum_f0", "cum", 0), ("pair_f0", "pair", 0), ("cum_f1", "cum", 1), ("cum_f2", "cum", 2)]\nCAP_ATTN = ("lr1e-4_r16_attn", "lr1e-4_r16")\n\n\ndef chon_mau_sinh(mau_cum, mau_nhanh, fold, moi_nhom, seed, rut_gon=None):\n    """Mẫu sinh cố định của một fold: chọn trên nhánh cụm đúng như B6 (seed + 1), rồi lấy cùng id ở nhánh cần chấm, để hai nhánh chấm trên cùng sản phẩm.\n    rut_gon: giữ vài mẫu đầu mỗi nhóm khi chạy khói, vẫn là tập con của mẫu thật nên dùng lại được bản dịch đã có. Trả (mẫu, số id thiếu ở nhánh đó)."""\n    chon = lay_phan_tang([m for m in mau_cum if m["fold5"] == fold and m["phan"] == 0], moi_nhom, seed + 1)\n    if rut_gon:\n        dem, gon = {}, []\n        for m in chon:\n            k = f"{m[\'huong\']}|{m[\'loai\']}"\n            dem[k] = dem.get(k, 0) + 1\n            if dem[k] <= rut_gon:\n                gon.append(m)\n        chon = gon\n    theo = {m["id"]: m for m in mau_nhanh}\n    return [theo[m["id"]] for m in chon if m["id"] in theo], sum(m["id"] not in theo for m in chon)\n\n\ndef gop_loss(tung_cat):\n    """tung_cat {\'huong|loai|category\': (tổng loss, số token)}. Trả (loss {\'huong|loai\'} cộng theo token, điểm macro theo hướng)."""\n    gop = {}\n    for k, (t, n) in tung_cat.items():\n        a = gop.setdefault(k.rsplit("|", 1)[0], [0.0, 0])\n        a[0] += t\n        a[1] += n\n    nhom = {k: t / max(n, 1) for k, (t, n) in sorted(gop.items())}\n    return nhom, diem_macro(nhom)\n\n\n# Tỉ lệ title : mô tả (mục tiêu d, plan mục 0). Trọng số đặt theo mẫu: mô tả = 1, title = w. Một micro-batch: token nhân trọng số mẫu, chia tổng trọng số token.\n# Một bước: micro-batch nặng theo (tổng trọng số mẫu / số mẫu của bước) nhân hệ số chung he_chuan, KHÔNG chia cho tổng trọng số của bước: nhiều bước chỉ toàn title\n# hoặc toàn mô tả, chia theo bước sẽ triệt tiêu trọng số ở đó (đo thật: tỉ trọng title kẹt trong 14%–71%). w = 1 trùng đúng cách tính không trọng số.\ndef tom_tat_buoc(ds_buoc):\n    """Rút gọn để tính nhanh: mỗi bước là danh sách (số mẫu title, số mẫu mô tả, token đích title, token đích mô tả) từng micro-batch."""\n    ra = []\n    for buoc in ds_buoc:\n        hang = []\n        for k in buoc:\n            n_t = n_m = t_t = t_m = 0\n            for m in k:\n                so = sum(1 for x in m["labels"] if x != -100)\n                if m["loai"] == "title":\n                    n_t, t_t = n_t + 1, t_t + so\n                else:\n                    n_m, t_m = n_m + 1, t_m + so\n            hang.append((n_t, n_m, t_t, t_m))\n        ra.append(hang)\n    return ra\n\n\ndef ti_trong_title(tom_tat, w):\n    """Phần trọng số title trong loss, cộng trên mọi bước; là tỉ lệ danh nghĩa khi đặt w, chưa tính tác động của clip gradient."""\n    title = tong = 0.0\n    for hang in tom_tat:\n        so_mau = sum(a + b for a, b, _, _ in hang)\n        for a, b, c, d in hang:\n            nang = (w * a + b) / so_mau\n            mau = w * c + d\n            tong += nang\n            if mau:\n                title += nang * (w * c / mau)\n    return title / tong\n\n\ndef he_chuan(mau, w):\n    """Giữ độ lớn loss trung bình như khi không trọng số, để lr vẫn so được với mốc."""\n    return len(mau) / sum(trong_so_lo(mau, w))\n\n\ndef tim_w_title(tom_tat, p, so_lan=60):\n    """Tìm w để tỉ trọng title đạt p, chia đôi trên thang log vì ti_trong_title tăng theo w."""\n    if not 0 < p < 1:\n        raise ValueError(f"p_ngoai_khoang:{p}")\n    thap, cao = math.log(1e-4), math.log(1e4)\n    for _ in range(so_lan):\n        giua = (thap + cao) / 2\n        if ti_trong_title(tom_tat, math.exp(giua)) < p:\n            thap = giua\n        else:\n            cao = giua\n    return math.exp((thap + cao) / 2)\n\n\ndef trong_so_lo(k, w):\n    """Trọng số từng mẫu của micro-batch: title = w, mô tả = 1."""\n    return [w if m["loai"] == "title" else 1.0 for m in k]\n\n\ndef kiem_bat_thuong(cham, chrf_toi_thieu, lap_toi_da):\n    """Dấu hiệu lần chạy hỏng, để dừng sớm thay vì đốt tiếp quota khi không ai theo dõi. Trả lý do hoặc None. Ngưỡng là để bắt hỏng, không phải để chọn tỉ lệ."""\n    diem = cham["chrf"]["chrf_macro_huong"]\n    if diem is None or diem < chrf_toi_thieu:\n        return f"chrF {diem} duoi nguong {chrf_toi_thieu}"\n    ti_le = cham["sinh_lap"] / max(cham["so_mau"], 1)\n    if ti_le > lap_toi_da:\n        return f"sinh lap {ti_le:.3f} vuot nguong {lap_toi_da}"\n    return None\n\n\ndef chon_trang_thai(ds):\n    """ds: [(đường dẫn, trạng thái)]. Khi gắn cả bản lưu và đầu ra chính notebook, lấy bản tiến xa hơn (nhiều bước và nhiều phần chấm hơn), tránh nối nhầm bản cũ."""\n    def tien_do(x):\n        viec = x[1].get("viec", {}).values()\n        return (sum(1 for v in viec if v.get("cham")), sum(v.get("buoc", 0) for v in viec))\n    return max(ds, key=tien_do)\n\n\n# ---------------------------------------------------------------- môi trường\ndef tim_mot(mau_ten):\n    ds = sorted(VAO.rglob(mau_ten))\n    if not ds:\n        raise RuntimeError(f"khong_thay:{mau_ten}")\n    return ds[0]\n\n\ndef kiem_khoa_test():\n    la = [str(p) for p in VAO.rglob("*") if p.is_file() and ("test_" in p.name or p.name.startswith("do_khoa_") or p.name in {"split.parquet", "ds_test.txt"})\n          and "site-packages" not in str(p) and "arc-prize" not in str(p)]\n    if la:\n        raise RuntimeError(f"CO_FILE_TEST_TRONG_INPUT:{la[:5]}")\n\n\ndef kiem_khoa_test_do():\n    """Chế độ đo test (Nhật Anh duyệt 07/11/2026): được phép có file do_khoa_*.jsonl*; mọi file \'test_*\' khác, split.parquet, ds_test.txt vẫn là lỗi.\n    Phải có ít nhất một file do_khoa_*. Việc không rò sang pool kiểm riêng bằng kiem_ro_ri_test."""\n    la = [p for p in VAO.rglob("*") if p.is_file() and "site-packages" not in str(p) and "arc-prize" not in str(p)]\n    cam = [str(p) for p in la if p.name in {"split.parquet", "ds_test.txt"} or ("test_" in p.name and not p.name.startswith("do_khoa_"))]\n    if cam:\n        raise RuntimeError(f"CO_FILE_TEST_LA_TRONG_INPUT:{cam[:5]}")\n    if not any(p.name.startswith("do_khoa_") for p in la):\n        raise RuntimeError("THIEU_DO_KHOA")\n\n\ndef kiem_ro_ri_test(mau_test, mau_pool):\n    """Không sản phẩm hay id nào của test nằm trong pool (cùng nhánh). Trả (số sp test, số mẫu test)."""\n    sp_pool, id_pool = {m["product_id"] for m in mau_pool}, {m["id"] for m in mau_pool}\n    chung_sp = {m["product_id"] for m in mau_test} & sp_pool\n    chung_id = {m["id"] for m in mau_test} & id_pool\n    if chung_sp or chung_id:\n        raise RuntimeError(f"TEST_ROI_VAO_POOL:{len(chung_sp)}sp:{len(chung_id)}id")\n    return len({m["product_id"] for m in mau_test}), len(mau_test)\n\n\ndef cai_thu_vien():\n    wh = tim_mot("peft-*.whl").parent\n    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", str(wh), "peft", "sacrebleu"],\n                       capture_output=True, text=True)\n    print((r.stdout or "")[-1500:], (r.stderr or "")[-1500:], flush=True)\n    if r.returncode:\n        raise RuntimeError("cai_thu_vien_loi")\n\n\ndef doc_mau(nhanh):\n    """Kaggle có thể tự giải nén .gz khi nhận Dataset, nên đọc được cả hai dạng (nhận theo chữ ký file, không theo đuôi)."""\n    ds = [p for p in sorted(VAO.rglob(f"pool_{nhanh}.jsonl*")) if p.is_file()]\n    if not ds:\n        raise RuntimeError(f"khong_thay:pool_{nhanh}.jsonl")\n    with open(ds[0], "rb") as f:\n        nen = f.read(2) == b"\\x1f\\x8b"\n    print("doc mau:", ds[0], "nen" if nen else "khong nen", flush=True)\n    mo = gzip.open(ds[0], "rt", encoding="utf-8") if nen else open(ds[0], encoding="utf-8")\n    with mo as f:\n        return [json.loads(x) for x in f]\n\n\n# ---------------------------------------------------------------- phần cần torch\ndef lo_tensor(lo, tok, thiet_bi):\n    import torch\n    dai = max(len(m["ids"]) for m in lo)\n    pad = tok.pad_token_id\n    ids = torch.tensor([m["ids"] + [pad] * (dai - len(m["ids"])) for m in lo], device=thiet_bi)\n    lab = torch.tensor([m["labels"] + [-100] * (dai - len(m["labels"])) for m in lo], device=thiet_bi)\n    mask = torch.tensor([[1] * len(m["ids"]) + [0] * (dai - len(m["ids"])) for m in lo], device=thiet_bi)\n    return ids, lab, mask\n\n\ndef loss_kiem(model, tok, mau, micro_bs, thiet_bi, ngan_sach=8192):\n    """Loss trung bình theo token đích trên tập kiểm cố định."""\n    tong, dem = loss_tong(model, tok, mau, micro_bs, thiet_bi, ngan_sach)\n    return tong / max(dem, 1)\n\n\ndef loss_tong(model, tok, mau, micro_bs, thiet_bi, ngan_sach=8192):\n    """Tổng loss và số token đích, để cộng dồn đúng trọng số giữa các nhóm."""\n    import torch\n    import torch.nn.functional as F\n    model.eval()\n    tong, dem = 0.0, 0\n    with torch.no_grad():\n        for k in cat_ngan_sach(sorted(mau, key=lambda m: len(m["ids"])), micro_bs, ngan_sach):\n            ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n            logit = model(input_ids=ids, attention_mask=mask).logits[:, :-1]\n            nhan = lab[:, 1:]\n            tong += F.cross_entropy(logit.reshape(-1, logit.size(-1)).float(), nhan.reshape(-1), ignore_index=-100, reduction="sum").item()\n            dem += (nhan != -100).sum().item()\n    model.train()\n    return tong, dem\n\n\ndef loss_co_trong_so(model, ids, lab, mask, ts, thiet_bi):\n    """Loss một micro-batch: token nhân trọng số mẫu, chia tổng trọng số token. ts toàn 1 thì bằng out.loss của mô hình (có phép thử trước khi chạy)."""\n    import torch\n    import torch.nn.functional as F\n    logit = model(input_ids=ids, attention_mask=mask).logits[:, :-1].float()\n    nhan = lab[:, 1:]\n    ce = F.cross_entropy(logit.reshape(-1, logit.size(-1)), nhan.reshape(-1), ignore_index=-100, reduction="none").view_as(nhan)\n    tw = torch.tensor(ts, dtype=torch.float32, device=thiet_bi)[:, None] * (nhan != -100)\n    return (ce * tw).sum() / tw.sum()\n\n\ndef nap_goc(duong_dan, attn="sdpa"):\n    """attn mặc định sdpa như mọi lần chạy cũ (Qwen3); mô hình BLOOM (EcomGPT) không dùng sdpa nên truyền attn="eager"."""\n    import torch\n    from transformers import AutoModelForCausalLM\n    return AutoModelForCausalLM.from_pretrained(str(duong_dan), dtype=torch.bfloat16, device_map={"": 0}, attn_implementation=attn)\n\n\ndef chon_tran_sinh(n_tok_nguon):\n    """Trần max_new_tokens theo độ dài NGUỒN (đo trên pool 07/11/2026: token đích / token nguồn lớn nhất 2,94 ở title zh->vi, 1,59 ở mô tả), nên 3,5 lần cộng 64.\n    Dùng khi đo test, vì lúc dịch thật không có tham chiếu để lấy độ dài."""\n    return min(2048, math.ceil(3.5 * n_tok_nguon) + 64)\n\n\ndef sinh(model, tok, mau, thiet_bi, bs=32, ngan_sach=32768, tran_theo="dich", lam_prompt=None):\n    """Giải mã greedy cố định; trần max_new_tokens theo độ dài đích tham chiếu (tran_theo="dich", mặc định, giữ nguyên cho mọi lần chạy đã làm)\n    hoặc theo độ dài nguồn (tran_theo="nguon", dùng khi đo test). Xếp theo độ dài và cắt lô theo ngân sách cho nhanh; trả đúng thứ tự mau."""\n    import torch\n    tok.padding_side = "left"\n    model.eval()\n    mo = []\n    for m in mau:\n        # lam_prompt=None: khuôn chat Qwen3 như mọi lần chạy cũ; mô hình khác truyền hàm dựng lời dẫn riêng (xem kaggle_quota.tao_khuon)\n        p = lam_prompt(m) if lam_prompt else tok.apply_chat_template(m["messages"][:1], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n        if tran_theo == "nguon":\n            tran = chon_tran_sinh(len(tok(m["nguon"], add_special_tokens=False)["input_ids"]))\n        else:\n            tran = min(1500, len(tok(m["dich"], add_special_tokens=False)["input_ids"]) * 2 + 64)\n        mo.append({"m": m, "p": p, "tran": tran, "ids": [0] * (len(tok(p, add_special_tokens=False)["input_ids"]) + tran)})\n    kq = {}\n    for k in cat_ngan_sach(sorted(mo, key=lambda x: len(x["ids"])), bs, ngan_sach):\n        enc = tok([x["p"] for x in k], return_tensors="pt", padding=True, add_special_tokens=False).to(thiet_bi)\n        with torch.no_grad():\n            out = model.generate(**enc, max_new_tokens=max(x["tran"] for x in k), do_sample=False, pad_token_id=tok.pad_token_id)\n        for x, o in zip(k, out):\n            kq[x["m"]["id"]] = tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True)\n    tok.padding_side = "right"\n    return [{"id": m["id"], "hyp_tho": kq[m["id"]], "hyp": kq[m["id"]]} for m in mau]\n\n\ndef loss_nhom(model, tok, mau, thiet_bi, ngan_sach=8192):\n    """Loss theo token đích của từng nhóm \'huong|loai\' trên mẫu kiểm cố định; trả dict và điểm macro."""\n    theo = {}\n    for m in mau:\n        theo.setdefault(f"{m[\'huong\']}|{m[\'loai\']}", []).append(m)\n    ra = {k: loss_kiem(model, tok, v, 32, thiet_bi, ngan_sach) for k, v in sorted(theo.items())}\n    return ra, diem_macro(ra)\n\n\ndef chay_b6(cfg, bc):\n    """B6 dò loại dần trên 1 fold (Nhật Anh chốt 05/11/2026). Vòng 1: mọi cấu hình 1 epoch, so loss. Vòng 2: cấu hình được chọn chạy nốt tới epoch cuối (SO_EPOCH_B6).\n    Vòng 3: cấu hình được chọn, ở epoch loss tốt nhất, sinh và chấm chrF trên mẫu cố định. Lịch lr cosine tính cho toàn bộ SO_EPOCH_B6 epoch nên vòng 2 nối tiếp đúng lịch.\n    Trạng thái ghi vào b6/b6_trang_thai.json sau mỗi mốc; hết giờ thì lưu rồi dừng có trật tự, phiên sau chạy tiếp."""\n    import shutil\n    import torch\n    import transformers\n    import peft\n    from peft import LoraConfig, PeftModel, get_peft_model\n    from transformers import AutoTokenizer\n\n    thiet_bi = "cuda:0"\n    t_bat_dau = cfg["_t_bat_dau"]\n    het_giay = cfg.get("gio_toi_da", 8.5) * 3600\n    thu_ra = RA / "b6"\n    tep = thu_ra / "b6_trang_thai.json"\n    if not tep.exists():                               # phiên sau: gắn trạng thái phiên trước làm input\n        cu = sorted(VAO.rglob("b6_trang_thai.json"))\n        if cu:\n            shutil.copytree(cu[0].parent, thu_ra)\n            print("chay tiep tu:", cu[0], flush=True)\n    thu_ra.mkdir(exist_ok=True)\n    st = json.loads(tep.read_text(encoding="utf-8")) if tep.exists() else {"cau_hinh": {}, "chon": None, "cham": {}}\n\n    def ghi():\n        tam = tep.with_suffix(".tmp")\n        tam.write_text(json.dumps(st, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n        tam.replace(tep)\n        bc["b6"] = st\n\n    bc["moi_truong"] = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__, "gpu": torch.cuda.get_device_name(0)}\n    thu_mh = tim_mot("REVISION.txt").parent\n    bc["mo_hinh"] = thu_mh.joinpath("REVISION.txt").read_text().split()\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    goc_mau = doc_mau(cfg["nhanh"])\n    ma, bo = xay_mau(goc_mau, tok, cfg["max_len"])\n    fk = cfg["fold_kiem"]\n    train = [m for m in ma if m["fold5"] != fk]\n    kiem = lay_phan_tang([m for m in ma if m["fold5"] == fk], cfg["so_kiem_moi_nhom"], cfg["seed"])\n    if cfg.get("so_train"):\n        train = random.Random(cfg["seed"]).sample(train, cfg["so_train"])\n    if {m["id"] for m in train} & {m["id"] for m in kiem}:\n        raise RuntimeError("giao_train_kiem")\n    st.update(so_train=len(train), so_kiem=len(kiem), so_mau_bo_max_len=bo, so_token_train=sum(len(m["ids"]) for m in train), fold_kiem=fk)\n    co_batch = cfg["micro_bs"] * cfg["tich_luy"]\n    # Chống nối nhầm trạng thái: phiên sau phải cùng dữ liệu, cùng cách chia lô, cùng số epoch với phiên đã tạo trạng thái\n    van_tay = {"so_train": len(train), "so_kiem": len(kiem), "so_token_train": st["so_token_train"], "fold_kiem": fk, "seed": cfg["seed"], "so_epoch": SO_EPOCH_B6,\n               "micro_bs": cfg["micro_bs"], "ngan_sach": cfg["ngan_sach_token"], "nhanh": cfg["nhanh"]}\n    if st.get("van_tay") and st["van_tay"] != van_tay:\n        raise RuntimeError(f"TRANG_THAI_KHONG_KHOP: cu {st[\'van_tay\']} moi {van_tay}")\n    st["van_tay"] = van_tay\n    # Bước của từng epoch được dựng trước, cùng seed cho mọi cấu hình; lịch lr dùng đúng tổng số bước của mọi epoch\n    buoc_theo_epoch = [gom_buoc(chia_lo(train, cfg["micro_bs"], cfg["seed"] + e, cfg["ngan_sach_token"]), co_batch) for e in range(SO_EPOCH_B6)]\n    tong_buoc = sum(len(x) for x in buoc_theo_epoch)\n    st["tong_buoc"] = tong_buoc\n    ds_cau_hinh = {c["ten"]: c for c in CAU_HINH_B6 if not cfg.get("chi_cau_hinh") or c["ten"] in cfg["chi_cau_hinh"]}\n    for ten in ds_cau_hinh:\n        st["cau_hinh"].setdefault(ten, {"epoch_xong": 0, "buoc": 0, "vi_tri_lo": 0, "loss": {}, "diem": {}, "token_moi_giay": {}})\n    ghi()\n    goc = nap_goc(thu_mh)\n    goc.config.use_cache = False\n    dem_dung = {"con": cfg.get("dung_sau_buoc")}       # chỉ dùng khi chạy khói, để thử chạy tiếp\n\n    def het_gio():\n        return time.time() - t_bat_dau > het_giay\n\n    def huan_luyen_toi(ten, den_epoch):\n        nonlocal goc\n        c, s = ds_cau_hinh[ten], st["cau_hinh"][ten]\n        if s["epoch_xong"] >= den_epoch or s.get("hong"):\n            return\n        d, moi_nhat = thu_ra / ten, thu_ra / ten / "moi_nhat"\n        d.mkdir(exist_ok=True)\n        torch.manual_seed(cfg["seed"])\n        if (moi_nhat / "opt.pt").exists():\n            m = PeftModel.from_pretrained(goc, str(moi_nhat), is_trainable=True)\n        else:\n            m = get_peft_model(goc, LoraConfig(r=c["rank"], lora_alpha=2 * c["rank"], lora_dropout=0.05, target_modules=c.get("muc_tieu", "all-linear"), task_type="CAUSAL_LM"))\n        tham_so = [p for p in m.parameters() if p.requires_grad]\n        opt = torch.optim.AdamW(tham_so, lr=c["lr"], weight_decay=0.0)\n        if (moi_nhat / "opt.pt").exists():\n            opt.load_state_dict(torch.load(moi_nhat / "opt.pt", map_location=thiet_bi))\n        s["so_tham_so"] = sum(p.numel() for p in tham_so)\n\n        def luu():\n            if moi_nhat.exists():\n                shutil.rmtree(moi_nhat)\n            m.save_pretrained(str(moi_nhat))\n            torch.save(opt.state_dict(), moi_nhat / "opt.pt")\n            ghi()\n\n        m.train()\n        hong = False\n        for e in range(s["epoch_xong"], den_epoch):\n            ds_buoc = buoc_theo_epoch[e]\n            i, tok_do, t0 = s["vi_tri_lo"], 0, time.time()\n            while i < len(ds_buoc) and not hong:\n                cac_lo = ds_buoc[i]\n                i += 1\n                so_mau = sum(len(k) for k in cac_lo)\n                tong = 0.0\n                for k in cac_lo:\n                    ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n                    out = m(input_ids=ids, attention_mask=mask, labels=lab)\n                    (out.loss * len(k) / so_mau).backward()         # trọng số theo số mẫu thật của bước\n                    tong += out.loss.item() * len(k) / so_mau\n                    tok_do += int(mask.sum().item())\n                for g in opt.param_groups:\n                    g["lr"] = c["lr"] * lich_lr(s["buoc"], tong_buoc)\n                torch.nn.utils.clip_grad_norm_(tham_so, 1.0)\n                opt.step()\n                opt.zero_grad(set_to_none=True)\n                s["buoc"] += 1\n                s["vi_tri_lo"] = i\n                if not math.isfinite(tong):\n                    hong = True\n                if s["buoc"] % 100 == 0:\n                    print(ten, "epoch", e + 1, "buoc", s["buoc"], "/", tong_buoc, "loss", round(tong, 4), "lr", opt.param_groups[0]["lr"], flush=True)\n                if dem_dung["con"] is not None:\n                    dem_dung["con"] -= 1\n                dung = dem_dung["con"] == 0\n                if not hong and (het_gio() or dung):\n                    if dung:\n                        dem_dung["con"] = None\n                    luu()\n                    print("DUNG, da luu", ten, "epoch", e + 1, "vi tri lo", i, flush=True)\n                    goc = m.unload()           # gỡ LoRA khỏi mô hình gốc, tránh chồng hai lớp khi chạy tiếp trong cùng phiên\n                    del m, opt\n                    torch.cuda.empty_cache()\n                    raise DungHetGio(ten)\n            if hong:\n                s.update(hong=True)\n                s["diem"][str(e + 1)] = float("inf")\n                print("LOSS KHONG HUU HAN, bo cau hinh", ten, flush=True)\n                break\n            s["token_moi_giay"][str(e + 1)] = round(tok_do / max(time.time() - t0, 1e-6), 1)\n            s["epoch_xong"], s["vi_tri_lo"] = e + 1, 0\n            ln, dm = loss_nhom(m, tok, kiem, thiet_bi)\n            m.train()\n            s["loss"][str(e + 1)], s["diem"][str(e + 1)] = ln, dm\n            m.save_pretrained(str(d / f"ep{e + 1}"))\n            luu()\n            print(ten, "XONG epoch", e + 1, "diem loss", round(dm, 5), ln, "token/giay", s["token_moi_giay"][str(e + 1)], "gio da chay", round((time.time() - t_bat_dau) / 3600, 2), flush=True)\n        goc = m.unload()\n        del m, opt\n        torch.cuda.empty_cache()\n        ghi()\n\n    # Vòng 1\n    for ten in ds_cau_hinh:\n        huan_luyen_toi(ten, 1)\n    if st["chon"] is None:\n        st["chon"] = chon_vong2({t: s["diem"].get("1", float("inf")) for t, s in st["cau_hinh"].items()})\n        ghi()\n    print("CHON VONG 2:", st["chon"], flush=True)\n    # Vòng 2\n    for ten in st["chon"]:\n        huan_luyen_toi(ten, SO_EPOCH_B6)\n    # Vòng 3: sinh và chấm trên mẫu cố định, ở epoch loss tốt nhất của từng cấu hình\n    import pandas as pd\n    import cham_diem as D\n    mau_sinh = lay_phan_tang([m for m in goc_mau if m["fold5"] == fk and m["phan"] == 0], cfg["so_sinh_moi_nhom"], cfg["seed"] + 1)\n    tc = pd.DataFrame(mau_sinh)\n    diem_muc = {}\n    for ten in st["chon"]:\n        s = st["cau_hinh"][ten]\n        if s.get("hong"):\n            continue\n        ep = min(s["diem"], key=lambda e: s["diem"][e])\n        tep_sinh = thu_ra / ten / f"sinh_ep{ep}.jsonl"\n        if ten not in st["cham"]:\n            if het_gio():\n                raise DungHetGio("sinh")\n            m = PeftModel.from_pretrained(goc, str(thu_ra / ten / f"ep{ep}"))\n            t = time.time()\n            du = sinh(m, tok, mau_sinh, thiet_bi)\n            giay = time.time() - t\n            tep_sinh.write_text("\\n".join(json.dumps(r, ensure_ascii=False) for r in du) + "\\n", encoding="utf-8")\n            goc = m.unload()\n            del m\n            torch.cuda.empty_cache()\n            tm, th = D.cham(tc, pd.DataFrame(du), nhanh=cfg["nhanh"])\n            st["cham"][ten] = {"epoch": ep, "giay_moi_mau": round(giay / len(du), 3), "chrf": th["diem_chinh"], "theo_nhom": th.get("theo_nhom"),\n                               "ti_le_cat_loi_dan": float(tm.da_cat_loi_dan.mean()), "sinh_lap": int(tm.sinh_lap.sum()) if "sinh_lap" in tm else None}\n            ghi()\n            print(ten, "epoch", ep, "chrF", th["diem_chinh"].get("chrf_macro_huong"), "giay/mau", st["cham"][ten]["giay_moi_mau"], flush=True)\n        du = [json.loads(x) for x in tep_sinh.read_text(encoding="utf-8").splitlines()]\n        tm, _ = D.cham(tc, pd.DataFrame(du), nhanh=cfg["nhanh"])\n        diem_muc[ten] = tm.set_index("id").chrf\n    # So theo cặp: cấu hình đầu bảng theo chrF với từng cấu hình còn lại, khoảng tin cậy bootstrap (chứa 0 nghĩa là hòa)\n    xep = sorted(st["cham"], key=lambda t: -st["cham"][t]["chrf"]["chrf_macro_huong"])\n    nhom = [f"{m[\'huong\']}|{m[\'loai\']}" for m in mau_sinh]\n    ids = [m["id"] for m in mau_sinh]\n    st["so_cap_chrf"] = {f"{xep[0]} - {t}": bootstrap_chenh(diem_muc[xep[0]][ids].tolist(), diem_muc[t][ids].tolist(), nhom) for t in xep[1:]}\n    st["xep_hang_chrf"] = xep\n    st["xong"] = True\n    ghi()\n    bc["dat"] = True\n    print("XEP HANG CHRF:", xep, st["so_cap_chrf"], flush=True)\n\n\ndef chay_b78(cfg, bc):\n    """B7, B8 (D18): cấu hình đã chốt chạy trên pair fold 0 và cụm fold 1, 2; cụm fold 0 dùng lại adapter và bản dịch B6. Mỗi việc: huấn luyện, loss cả fold theo\n    hướng, loại, category, rồi sinh và chấm chrF trên 1.000 mẫu cố định. Trước đó chấm chrF LoRA attention so với LoRA mọi lớp cùng lr, rank ở epoch 1.\n    Không tự chạy thêm khi chốt chặn bật: tong_hop_b78.py tính chốt chặn trên máy, người quyết. Không dùng chung vòng huấn luyện với chay_b6 để giữ nguyên mã đã tạo ra kết quả B6.\n    Mỗi pha tự bỏ qua nếu đã có trong trạng thái; hết giờ thì lưu rồi dừng có trật tự, phiên sau chạy tiếp."""\n    import shutil\n    import pandas as pd\n    import torch\n    import transformers\n    import peft\n    from peft import LoraConfig, PeftModel, get_peft_model\n    from transformers import AutoTokenizer\n    import cham_diem as D\n\n    thiet_bi = "cuda:0"\n    t_bat_dau = cfg["_t_bat_dau"]\n    het_giay = cfg["gio_toi_da"] * 3600\n    ten_tt = cfg.get("ten_trang_thai", "b78")      # khảo sát tỉ lệ dùng trạng thái riêng "tl", không lẫn với B7, B8\n    thu_ra = RA / ten_tt\n    tep = thu_ra / f"{ten_tt}_trang_thai.json"\n    if not tep.exists():\n        cu = sorted(VAO.rglob(tep.name))\n        if cu:\n            duong_cu, _ = chon_trang_thai([(p, json.loads(p.read_text(encoding="utf-8"))) for p in cu])\n            shutil.copytree(duong_cu.parent, thu_ra)\n            print("chay tiep tu:", duong_cu, flush=True)\n    thu_ra.mkdir(exist_ok=True)\n    st = json.loads(tep.read_text(encoding="utf-8")) if tep.exists() else {"viec": {}, "attn": {}, "xong": None}\n    ds_viec = cfg.get("viec") or [{"ten": t, "nhanh": n, "fold": f, "ca_fold": True} for t, n, f in VIEC_B78]\n\n    def ghi():\n        tam = tep.with_suffix(".tmp")\n        tam.write_text(json.dumps(st, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n        tam.replace(tep)\n        bc["b78"] = st\n\n    def kiem_gio(o_dau):\n        if time.time() - t_bat_dau > het_giay:\n            raise DungHetGio(o_dau)\n\n    # Chống nối nhầm trạng thái: phiên sau phải cùng cách chạy với phiên đã tạo trạng thái\n    van_tay = {k: cfg.get(k) for k in ("seed", "micro_bs", "ngan_sach_token", "max_len", "so_kiem_moi_nhom", "so_sinh_moi_nhom", "rut_gon_sinh", "so_train", "so_mau_ca_fold")}\n    van_tay.update(so_epoch=SO_EPOCH_B6, cau_hinh=CAU_HINH_CHOT["ten"])\n    if st.get("van_tay") and st["van_tay"] != van_tay:\n        raise RuntimeError(f"TRANG_THAI_KHONG_KHOP: cu {st[\'van_tay\']} moi {van_tay}")\n    st["van_tay"] = van_tay\n\n    kho, st_b6 = None, None\n    if not cfg.get("viec"):\n        ds_b6 = sorted(VAO.rglob("b6_trang_thai.json"))     # kho lưu B6, notebook riêng nên không bị thay khi đẩy lại nlp-group3\n        if len(ds_b6) != 1:\n            raise RuntimeError(f"can_dung_mot_kho_luu_b6:{ds_b6}")\n        kho = ds_b6[0].parent\n        st_b6 = json.loads(ds_b6[0].read_text(encoding="utf-8"))\n        print("kho luu B6:", kho, flush=True)\n    bc["moi_truong"] = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__, "gpu": torch.cuda.get_device_name(0)}\n    thu_mh = tim_mot("REVISION.txt").parent\n    bc["mo_hinh"] = thu_mh.joinpath("REVISION.txt").read_text().split()\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    goc_mau = {n: doc_mau(n) for n in ("cum", "pair")}\n    ma = {}\n    for n, g in goc_mau.items():\n        ma[n], bo = xay_mau(g, tok, cfg["max_len"])\n        st[f"so_mau_bo_max_len_{n}"] = bo\n    ghi()\n    goc = nap_goc(thu_mh)\n    goc.config.use_cache = False\n    dem_dung = {"con": cfg.get("dung_sau_buoc")}       # chỉ dùng khi chạy khói, để thử chạy tiếp\n\n    if any(j.get("p_title") for j in ds_viec):         # phép thử trước khi tốn giờ: trọng số 1:1 phải đúng bằng loss của mô hình, trên lô trộn title và mô tả\n        n_thu = ds_viec[0]["nhanh"]\n        tit = min((m for m in ma[n_thu] if m["loai"] == "title"), key=lambda m: len(m["ids"]))\n        mt = min((m for m in ma[n_thu] if m["loai"] == "mo_ta"), key=lambda m: len(m["ids"]))\n        ids, lab, mask = lo_tensor([tit, mt], tok, thiet_bi)\n        goc.eval()\n        with torch.no_grad():\n            a = goc(input_ids=ids, attention_mask=mask, labels=lab).loss.item()\n            b = loss_co_trong_so(goc, ids, lab, mask, [1.0, 1.0], thiet_bi).item()\n        goc.train()\n        st["thu_loss_1_1"] = {"out_loss": a, "co_trong_so": b}\n        print("thu loss 1:1", a, b, flush=True)\n        if not abs(a - b) <= 1e-3:         # viết thế này để NaN cũng bị bắt\n            raise RuntimeError(f"LOSS_TRONG_SO_KHONG_KHOP:{a}:{b}")\n\n    def dung_bat_thuong(ten, ly_do):\n        st["bat_thuong"] = f"{ten}: {ly_do}"\n        ghi()\n        print("BAT THUONG, dung phien de khong dot quota:", st["bat_thuong"], flush=True)\n        raise DungHetGio("bat_thuong")\n\n    def lay_mau_sinh(nhanh, fold):\n        mau, thieu = chon_mau_sinh(goc_mau["cum"], goc_mau[nhanh], fold, cfg["so_sinh_moi_nhom"], cfg["seed"], cfg.get("rut_gon_sinh"))\n        return mau, thieu\n\n    def tap_da_thay(nhanh, fold):\n        return D.xay_tap_da_thay(pd.DataFrame(goc_mau[nhanh])[["loai", "category", "nguon", "nhanh", "fold5"]], fold, nhanh)\n\n    def doc_sinh(duong_dan, mau):\n        ids = {m["id"] for m in mau}\n        du = [r for r in map(json.loads, duong_dan.read_text(encoding="utf-8").splitlines()) if r["id"] in ids]\n        if len(du) != len(mau):\n            raise RuntimeError(f"ban_dich_thieu_muc:{duong_dan}:{len(du)}/{len(mau)}")\n        return du\n\n    def ghi_sinh(duong_dan, du):\n        tam = duong_dan.with_suffix(".tmp")\n        tam.write_text("\\n".join(json.dumps(r, ensure_ascii=False) for r in du) + "\\n", encoding="utf-8")\n        tam.replace(duong_dan)\n\n    def cham_mot(nhanh, mau, du, tap, epoch, giay_moi_mau):\n        tm, th = D.cham(pd.DataFrame(mau), pd.DataFrame(du), nhanh=nhanh, tap_da_thay=tap)\n        return tm, D._sach({"epoch": str(epoch), "giay_moi_mau": giay_moi_mau, "so_mau": len(mau), "sinh_lap": int(tm.sinh_lap.sum()), "ti_le_cat_loi_dan": float(tm.da_cat_loi_dan.mean()),\n                            "chrf": th["diem_chinh"], "theo_huong": th["theo_huong"], "theo_nhom": th["theo_nhom"], "theo_category": th["theo_category"], "dong_da_thay_moi": th.get("dong_da_thay_moi")})\n\n    def huan_luyen(ten, v, nhanh, fold, p_title=None, lr=None, loai_sp=None):\n        """fold là số: huấn luyện trên các fold khác và kiểm trên fold đó (như mọi lần chạy trước). fold là None: BẢN CUỐI trên TOÀN pool (Nhật Anh duyệt 07/11/2026, Prompt 808);\n        loai_sp (tập product_id) = bỏ mẫu title của các sản phẩm đó khỏi tập huấn luyện, giữ nguyên mẫu mô tả; mẫu \'kiểm\' lấy từ chính tập huấn luyện nên CHỈ để bắt hỏng, không phải đo chất lượng."""\n        nonlocal goc\n        if v["epoch_xong"] >= SO_EPOCH_B6 or v.get("hong"):\n            return\n        if fold is None:\n            tat_ca = [m for m in ma[nhanh] if not (loai_sp and m["loai"] == "title" and m["product_id"] in loai_sp)]\n            v.update(toan_pool=True, so_mau_pool=len(ma[nhanh]), so_title_bi_loai=len(ma[nhanh]) - len(tat_ca), kiem_la_train=True)\n            train = tat_ca\n        else:\n            train = [m for m in ma[nhanh] if m["fold5"] != fold]\n        if cfg.get("so_train"):\n            train = random.Random(cfg["seed"]).sample(train, cfg["so_train"])\n        if fold is None:\n            kiem = lay_phan_tang(train, cfg["so_kiem_moi_nhom"], cfg["seed"])\n        else:\n            kiem = lay_phan_tang([m for m in ma[nhanh] if m["fold5"] == fold], cfg["so_kiem_moi_nhom"], cfg["seed"])\n            if {m["id"] for m in train} & {m["id"] for m in kiem}:\n                raise RuntimeError("giao_train_kiem")\n        co_batch = cfg["micro_bs"] * cfg["tich_luy"]\n        buoc_theo_epoch = [gom_buoc(chia_lo(train, cfg["micro_bs"], cfg["seed"] + e, cfg["ngan_sach_token"]), co_batch) for e in range(SO_EPOCH_B6)]\n        tong_buoc = sum(len(x) for x in buoc_theo_epoch)\n        if v.get("tong_buoc") not in (None, tong_buoc):\n            raise RuntimeError(f"TONG_BUOC_KHONG_KHOP:{ten}:{v[\'tong_buoc\']}:{tong_buoc}")\n        v.update(so_train=len(train), so_kiem=len(kiem), so_token_train=sum(len(m["ids"]) for m in train), tong_buoc=tong_buoc)\n        w = None\n        if p_title:\n            tat = tom_tat_buoc([b for e in buoc_theo_epoch for b in e])\n            w = tim_w_title(tat, p_title)\n            if v.get("w_title") is not None and abs(v["w_title"] - w) > 1e-6:\n                raise RuntimeError(f"W_TITLE_KHONG_KHOP:{ten}:{v[\'w_title\']}:{w}")\n            chuan = he_chuan(train, w)\n            v.update(p_muc_tieu=p_title, w_title=w, he_chuan=chuan, p_thuc=ti_trong_title(tat, w), p_khi_w_1=ti_trong_title(tat, 1.0))\n            print(ten, "p muc tieu", p_title, "p thuc", round(v["p_thuc"], 5), "w_title", round(w, 5), "he_chuan", round(chuan, 4), "p khi w=1", round(v["p_khi_w_1"], 5), flush=True)\n            if abs(v["p_thuc"] - p_title) > 1e-3:\n                raise RuntimeError(f"KHONG_DAT_TI_LE:{ten}:{v[\'p_thuc\']}:{p_title}")\n        ghi()\n        c = {**CAU_HINH_CHOT, "lr": lr or CAU_HINH_CHOT["lr"]}\n        v["lr"] = c["lr"]\n        d, moi_nhat = thu_ra / ten, thu_ra / ten / "moi_nhat"\n        d.mkdir(exist_ok=True)\n        torch.manual_seed(cfg["seed"])\n        if v["buoc"] > 0 and not (moi_nhat / "opt.pt").exists():     # đã chạy dở mà mất checkpoint: dừng, không âm thầm tạo LoRA mới rồi chạy tiếp từ giữa\n            raise RuntimeError(f"MAT_CHECKPOINT:{ten}:buoc={v[\'buoc\']}")\n        if (moi_nhat / "opt.pt").exists():\n            m = PeftModel.from_pretrained(goc, str(moi_nhat), is_trainable=True)\n        else:\n            m = get_peft_model(goc, LoraConfig(r=c["rank"], lora_alpha=2 * c["rank"], lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"))\n        tham_so = [p for p in m.parameters() if p.requires_grad]\n        opt = torch.optim.AdamW(tham_so, lr=c["lr"], weight_decay=0.0)\n        if (moi_nhat / "opt.pt").exists():\n            opt.load_state_dict(torch.load(moi_nhat / "opt.pt", map_location=thiet_bi))\n        v["so_tham_so"] = sum(p.numel() for p in tham_so)\n\n        def luu():\n            if moi_nhat.exists():\n                shutil.rmtree(moi_nhat)\n            m.save_pretrained(str(moi_nhat))\n            torch.save(opt.state_dict(), moi_nhat / "opt.pt")\n            ghi()\n\n        m.train()\n        for e in range(v["epoch_xong"], SO_EPOCH_B6):\n            ds_buoc = buoc_theo_epoch[e]\n            i, tok_do, t0 = v["vi_tri_lo"], 0, time.time()\n            while i < len(ds_buoc):\n                cac_lo = ds_buoc[i]\n                i += 1\n                so_mau = sum(len(k) for k in cac_lo)\n                tong = 0.0\n                for k in cac_lo:\n                    ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n                    if w:\n                        ts = trong_so_lo(k, w)\n                        loss_k, he = loss_co_trong_so(m, ids, lab, mask, ts, thiet_bi), chuan * sum(ts) / so_mau\n                    else:\n                        loss_k, he = m(input_ids=ids, attention_mask=mask, labels=lab).loss, len(k) / so_mau\n                    (loss_k * he).backward()\n                    tong += loss_k.item() * he\n                    tok_do += int(mask.sum().item())\n                for g in opt.param_groups:\n                    g["lr"] = c["lr"] * lich_lr(v["buoc"], tong_buoc)\n                chuan_grad = float(torch.nn.utils.clip_grad_norm_(tham_so, 1.0))\n                # Clip có thể làm giảm tác dụng của trọng số: ghi tỉ lệ bước bị cắt theo loại để đánh giá, tách bước toàn title, toàn mô tả, trộn\n                nhom_buoc = {x["loai"] for k in cac_lo for x in k}\n                loai_buoc = "tron" if len(nhom_buoc) > 1 else next(iter(nhom_buoc))\n                tk = v.setdefault("thong_ke_grad", {}).setdefault(loai_buoc, {"so_buoc": 0, "tong_chuan": 0.0, "so_bi_cat": 0})\n                tk["so_buoc"] += 1\n                tk["tong_chuan"] += chuan_grad if math.isfinite(chuan_grad) else 0.0\n                tk["so_bi_cat"] += chuan_grad > 1.0\n                opt.step()\n                opt.zero_grad(set_to_none=True)\n                v["buoc"] += 1\n                v["vi_tri_lo"] = i\n                if not math.isfinite(tong):\n                    v["hong"] = True\n                    print("LOSS KHONG HUU HAN, bo viec", ten, "buoc", v["buoc"], flush=True)\n                    goc = m.unload()\n                    del m, opt\n                    torch.cuda.empty_cache()\n                    ghi()\n                    return\n                if v["buoc"] % 100 == 0:\n                    print(ten, "epoch", e + 1, "buoc", v["buoc"], "/", tong_buoc, "loss", round(tong, 4), "lr", opt.param_groups[0]["lr"],\n                          "grad", {a: (x["so_buoc"], round(x["tong_chuan"] / x["so_buoc"], 3), x["so_bi_cat"]) for a, x in v["thong_ke_grad"].items()}, flush=True)\n                if dem_dung["con"] is not None:\n                    dem_dung["con"] -= 1\n                dung = dem_dung["con"] == 0\n                if dung or time.time() - t_bat_dau > het_giay:\n                    if dung:\n                        dem_dung["con"] = None\n                    luu()\n                    print("DUNG, da luu", ten, "epoch", e + 1, "vi tri lo", i, flush=True)\n                    raise DungHetGio(ten)\n            v["token_moi_giay"][str(e + 1)] = round(tok_do / max(time.time() - t0, 1e-6), 1)\n            ln, dm = loss_nhom(m, tok, kiem, thiet_bi)\n            m.train()\n            v["loss"][str(e + 1)], v["diem"][str(e + 1)] = ln, dm\n            v["epoch_xong"], v["vi_tri_lo"] = e + 1, 0\n            if e + 1 == SO_EPOCH_B6:\n                m.save_pretrained(str(d / f"ep{e + 1}"))\n                ghi()\n                shutil.rmtree(moi_nhat, ignore_errors=True)    # xong việc thì bỏ, mỗi bản kèm opt.pt ~1,6 GB\n            else:\n                luu()\n            print(ten, "XONG epoch", e + 1, "diem loss", round(dm, 5), ln, "token/giay", v["token_moi_giay"][str(e + 1)], "gio da chay", round((time.time() - t_bat_dau) / 3600, 2), flush=True)\n        goc = m.unload()\n        del m, opt\n        torch.cuda.empty_cache()\n\n    def danh_gia(ten, v, nhanh, fold, thu_adapter, tep_sinh, ca_fold=True):\n        nonlocal goc\n        if v.get("hong") or ((v.get("loss_ca_fold") or not ca_fold) and v.get("cham")):\n            return\n        kiem_gio(ten)\n        m = PeftModel.from_pretrained(goc, str(thu_adapter))\n        if ca_fold and not v.get("loss_ca_fold"):\n            ca_fold = [x for x in ma[nhanh] if x["fold5"] == fold]\n            if cfg.get("so_mau_ca_fold"):\n                ca_fold = random.Random(cfg["seed"]).sample(ca_fold, cfg["so_mau_ca_fold"])\n            theo = {}\n            for x in ca_fold:\n                theo.setdefault(f"{x[\'huong\']}|{x[\'loai\']}|{x[\'category\']}", []).append(x)\n            t = time.time()\n            tung = {k: loss_tong(m, tok, a, 32, thiet_bi) for k, a in sorted(theo.items())}\n            nhom, dm = gop_loss(tung)\n            v["loss_ca_fold"] = {"so_mau": len(ca_fold), "diem_macro": dm, "theo_nhom": nhom, "theo_category": {k: {"loss": a / max(n, 1), "so_token": n} for k, (a, n) in tung.items()}}\n            ghi()\n            print(ten, "loss ca fold", round(dm, 5), nhom, "so mau", len(ca_fold), "giay", round(time.time() - t), flush=True)\n        if not v.get("cham"):\n            kiem_gio(ten)\n            mau, thieu = lay_mau_sinh(nhanh, fold)\n            v["so_id_sinh_thieu"] = thieu\n            if not tep_sinh.exists():\n                t = time.time()\n                du = sinh(m, tok, mau, thiet_bi)\n                v["giay_moi_mau"] = round((time.time() - t) / len(du), 3)\n                ghi_sinh(tep_sinh, du)\n            _, v["cham"] = cham_mot(nhanh, mau, doc_sinh(tep_sinh, mau), tap_da_thay(nhanh, fold), SO_EPOCH_B6, v.get("giay_moi_mau"))\n            ghi()\n            print(ten, "chrF", v["cham"]["chrf"]["chrf_macro_huong"], "so mau", len(mau), "thieu", thieu, "giay/mau", v.get("giay_moi_mau"), "gio da chay", round((time.time() - t_bat_dau) / 3600, 2), flush=True)\n        goc = m.unload()\n        del m\n        torch.cuda.empty_cache()\n\n    def kiem_nhanh_cuoi(ten, v, nhanh, thu_adapter):\n        """Bản cuối trên toàn pool: sinh thử vài mẫu (hướng x loại) CỦA CHÍNH TẬP HUẤN LUYỆN và chấm chrF, chỉ để bắt hỏng (đầu ra rỗng, lặp, chrF thấp bất thường).\n        Điểm này KHÔNG phải chất lượng trên dữ liệu mới; chất lượng dựa trên K-fold và lần đo test của adapter fold 0."""\n        nonlocal goc\n        if v.get("hong") or v.get("kiem_nhanh"):\n            return\n        kiem_gio(ten)\n        loai_sp = set(cfg.get("loai_title_sp") or []) if v.get("so_title_bi_loai") else set()\n        mau = lay_phan_tang([x for x in goc_mau[nhanh] if x["phan"] == 0 and not (x["loai"] == "title" and x["product_id"] in loai_sp)], cfg.get("so_sinh_kiem_nhanh", 25), cfg["seed"] + 7)\n        m = PeftModel.from_pretrained(goc, str(thu_adapter))\n        t = time.time()\n        du = sinh(m, tok, mau, thiet_bi)\n        giay = round((time.time() - t) / len(du), 3)\n        goc = m.unload()\n        del m\n        torch.cuda.empty_cache()\n        _, v["kiem_nhanh"] = cham_mot(nhanh, mau, du, None, SO_EPOCH_B6, giay)\n        v["kiem_nhanh"]["tren_du_lieu_huan_luyen"] = True\n        ghi()\n        print(ten, "KIEM NHANH (tren du lieu huan luyen) chrF", v["kiem_nhanh"]["chrf"]["chrf_macro_huong"], "so mau", len(mau), "sinh lap", v["kiem_nhanh"]["sinh_lap"], flush=True)\n\n    # Việc 1: LoRA attention so với LoRA mọi lớp cùng lr, rank, epoch 1, cụm fold 0, cùng mẫu sinh của B6. Khảo sát tỉ lệ không cần phần này\n    mau_a, tm_a = None, {}\n    if not cfg.get("viec"):\n        mau_a, _ = lay_mau_sinh("cum", 0)\n        (thu_ra / "attn").mkdir(exist_ok=True)\n    for ten in (() if cfg.get("viec") else CAP_ATTN):\n        tep_a = thu_ra / "attn" / f"{ten}_sinh.jsonl"\n        if ten not in st["attn"]:\n            kiem_gio(f"attn {ten}")\n            m = PeftModel.from_pretrained(goc, str(kho / ten / "ep1"))\n            t = time.time()\n            du = sinh(m, tok, mau_a, thiet_bi)\n            giay = round((time.time() - t) / len(du), 3)\n            ghi_sinh(tep_a, du)\n            goc = m.unload()\n            del m\n            torch.cuda.empty_cache()\n            _, st["attn"][ten] = cham_mot("cum", mau_a, du, None, 1, giay)\n            ghi()\n            print(ten, "epoch 1 chrF", st["attn"][ten]["chrf"]["chrf_macro_huong"], "giay/mau", giay, flush=True)\n        tm_a[ten], _ = cham_mot("cum", mau_a, doc_sinh(tep_a, mau_a), None, 1, None)\n    if not cfg.get("viec") and "so_cap" not in st["attn"]:\n        ids_a = [m["id"] for m in mau_a]\n        a, b = (tm_a[t].set_index("id").chrf[ids_a].tolist() for t in CAP_ATTN)\n        st["attn"]["so_cap"] = bootstrap_chenh(a, b, [f"{m[\'huong\']}|{m[\'loai\']}" for m in mau_a])\n        ghi()\n        print("ATTN - MOI LOP:", st["attn"]["so_cap"], flush=True)\n\n    # Việc 2 đến 5\n    ten_chot = CAU_HINH_CHOT["ten"]\n    for j in ds_viec:\n        ten, nhanh, fold = j["ten"], j["nhanh"], j["fold"]\n        v = st["viec"].setdefault(ten, {"nhanh": nhanh, "fold": fold, "cau_hinh": ten_chot, "epoch_xong": 0, "buoc": 0, "vi_tri_lo": 0, "loss": {}, "diem": {}, "token_moi_giay": {}, "loss_ca_fold": None, "cham": None})\n        if ten == "cum_f0":\n            b = st_b6["cau_hinh"][ten_chot]\n            v.update(nguon_adapter="b6", epoch_xong=b["epoch_xong"], loss=b["loss"], diem=b["diem"], token_moi_giay=b["token_moi_giay"], giay_moi_mau=st_b6["cham"][ten_chot]["giay_moi_mau"])\n            danh_gia(ten, v, nhanh, fold, kho / ten_chot / "ep2", kho / ten_chot / "sinh_ep2.jsonl")\n            if not cfg.get("rut_gon_sinh") and v.get("cham"):      # chấm lại bản dịch B6 phải ra đúng điểm B6\n                goc_b6 = st_b6["cham"][ten_chot]["chrf"]["chrf_macro_huong"]\n                v["khop_b6"] = abs(v["cham"]["chrf"]["chrf_macro_huong"] - goc_b6) < 1e-6\n                ghi()\n                if not v["khop_b6"]:\n                    raise RuntimeError(f"CHAM_LAI_KHONG_KHOP_B6:{v[\'cham\'][\'chrf\'][\'chrf_macro_huong\']}:{goc_b6}")\n            continue\n        huan_luyen(ten, v, nhanh, fold, j.get("p_title"), j.get("lr"), set(cfg.get("loai_title_sp") or []) if j.get("loai_title_co") else None)\n        if cfg.get("chrf_toi_thieu") is not None and v.get("hong"):\n            dung_bat_thuong(ten, "loss khong huu han")\n        if j.get("khong_danh_gia"):         # bản cuối trên toàn pool: không còn fold giữ lại; chỉ sinh thử vài mẫu để bắt hỏng\n            kiem_nhanh_cuoi(ten, v, nhanh, thu_ra / ten / f"ep{SO_EPOCH_B6}")\n            continue\n        danh_gia(ten, v, nhanh, fold, thu_ra / ten / f"ep{SO_EPOCH_B6}", thu_ra / ten / f"sinh_ep{SO_EPOCH_B6}.jsonl", j.get("ca_fold", True))\n        if cfg.get("chrf_toi_thieu") is not None and v.get("cham"):\n            ly_do = kiem_bat_thuong(v["cham"], cfg["chrf_toi_thieu"], cfg["lap_toi_da"])\n            if ly_do:\n                dung_bat_thuong(ten, ly_do)\n    st["xong"] = True\n    ghi()\n    bc["dat"] = not any(v.get("hong") for v in st["viec"].values())\n    print("B78 XONG", {t: (v["cham"] or {}).get("chrf", {}).get("chrf_macro_huong") for t, v in st["viec"].items()}, flush=True)\n\n\ndef chay_do_test(cfg, bc):\n    """Đo MỘT lần (B10, Nhật Anh duyệt 07/11/2026, Prompt 806): với mỗi nhánh (cụm, pair), sinh bản dịch của zero-shot (Qwen3-4B nguyên bản) và của adapter fold 0 ở ~11% title\n    trên cùng mẫu, trần sinh theo độ dài NGUỒN, rồi chấm bằng cham_diem. Không huấn luyện gì, không đổi cấu hình theo điểm.\n    cfg: nguon_do = "test" (đọc do_khoa_{nhanh}.jsonl* từ Dataset test riêng, kiểm không rò vào pool) hoặc "pool_fold" (CHỈ ĐỂ CHẠY KHÓI: lấy mẫu của một fold pool làm \'test\' giả,\n    không đụng test thật); mo_hinh_ft = {nhanh: tên việc trong kho lưu}; nhanh; gio_toi_da; so_mau_do (mỗi nhóm hướng x loại, chỉ khói).\n    Ra: do_test/do_test_trang_thai.json (điểm), do_test/sinh_{nhanh}_{mo_hinh}.jsonl (bản dịch), do_test/sinh_diem_{nhanh}_{mo_hinh}.jsonl (điểm từng mục, KHÔNG có tham chiếu).\n    Mỗi (nhánh, mô hình) tự bỏ qua nếu đã xong; hết giờ thì lưu rồi dừng có trật tự, phiên sau chạy tiếp."""\n    import gc\n    import hashlib\n    import shutil\n    import pandas as pd\n    import torch\n    import transformers\n    import peft\n    from peft import PeftModel\n    from transformers import AutoTokenizer\n    import cham_diem as D\n\n    thiet_bi = "cuda:0"\n    het_giay = cfg.get("gio_toi_da", 4.5) * 3600\n    thu_ra = RA / "do_test"\n    tep = thu_ra / "do_test_trang_thai.json"\n    if not tep.exists():\n        cu = sorted(VAO.rglob("do_test_trang_thai.json"))\n        if cu:\n            shutil.copytree(cu[0].parent, thu_ra)\n            print("chay tiep tu:", cu[0], flush=True)\n    thu_ra.mkdir(parents=True, exist_ok=True)\n    st = json.loads(tep.read_text(encoding="utf-8")) if tep.exists() else {"xong": False, "bat_thuong": None, "ket_qua": {}}\n    if st.get("nguon_do") not in (None, cfg["nguon_do"]):\n        raise RuntimeError(f"NOI_NHAM_NGUON_DO:{st.get(\'nguon_do\')}:{cfg[\'nguon_do\']}")          # không nối khói (mẫu pool) vào lần đo test thật hay ngược lại\n\n    def ghi():\n        tam = tep.with_suffix(".tmp")\n        tam.write_text(json.dumps(st, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n        tam.replace(tep)\n\n    bc["moi_truong"] = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__, "gpu": torch.cuda.get_device_name(0)}\n    thu_mh = tim_mot("REVISION.txt").parent\n    bc["mo_hinh"] = thu_mh.joinpath("REVISION.txt").read_text().split()\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    kho_tt = tim_mot("tl_trang_thai.json")\n    kho = kho_tt.parent\n    mf_kho = json.loads(tim_mot("MANIFEST_TL.json").read_text(encoding="utf-8"))          # nằm ngoài thư mục tl/ của kho lưu; khóa là đường dẫn tương đối so với tl/\n    st["nguon_do"] = cfg["nguon_do"]\n    st["mo_hinh_ft"] = cfg["mo_hinh_ft"]\n    st["cach_sinh"] = {"giai_ma": "greedy", "tran": "min(2048, ceil(3,5 x token nguon) + 64)", "enable_thinking": False}\n    ghi()\n\n    for nhanh in cfg["nhanh"]:\n        pool = doc_mau(nhanh)\n        if cfg["nguon_do"] == "test":\n            ds = [p for p in sorted(VAO.rglob(f"do_khoa_{nhanh}.jsonl*")) if p.is_file()]\n            if len(ds) != 1:\n                raise RuntimeError(f"do_khoa_{nhanh}: thay {len(ds)} file")\n            with open(ds[0], "rb") as f:\n                nen = f.read(2) == b"\\x1f\\x8b"\n            mo = gzip.open(ds[0], "rt", encoding="utf-8") if nen else open(ds[0], encoding="utf-8")\n            with mo as f:\n                mau = [json.loads(x) for x in f]\n            n_sp, n_mau = kiem_ro_ri_test(mau, pool)\n            st.setdefault("mau_do", {})[nhanh] = {"so_sp": n_sp, "so_mau": n_mau, "so_phan_khac_0": sum(1 for m in mau if m.get("phan", 0) != 0),\n                                                   "sha256": hashlib.sha256(ds[0].read_bytes()).hexdigest()}\n        else:\n            mau = lay_phan_tang([m for m in pool if m["fold5"] == cfg.get("fold_do", 4) and m["phan"] == 0], cfg.get("so_mau_do", 3), cfg["seed"])\n            st.setdefault("mau_do", {})[nhanh] = {"so_mau": len(mau), "ghi_chu": "KHOI: mau cua fold pool, khong phai test"}\n        for m in mau:\n            for k in ("id", "product_id", "huong", "loai", "nguon", "dich", "messages", "category"):\n                if k not in m:\n                    raise RuntimeError(f"mau_do_thieu_truong:{nhanh}:{k}")\n        # "đã thấy" = dòng nguồn có trong các fold mà adapter fold 0 đã học (fold khác 0), cùng cách với phần fold\n        tap = D.xay_tap_da_thay(pd.DataFrame(pool)[["loai", "category", "nguon", "nhanh", "fold5"]], cfg.get("fold_adapter", 0), nhanh)\n        ten_ft = cfg["mo_hinh_ft"][nhanh]\n        duong_ad = kho / ten_ft / "ep2"\n        rel = f"{ten_ft}/ep2/adapter_model.safetensors"\n        h = hashlib.sha256()\n        with open(duong_ad / "adapter_model.safetensors", "rb") as a:\n            for k in iter(lambda: a.read(2**20), b""):\n                h.update(k)\n        if h.hexdigest() != mf_kho[rel]["sha256"]:\n            raise RuntimeError(f"ADAPTER_LECH_HASH:{rel}")\n        st.setdefault("adapter", {})[nhanh] = {"ten": ten_ft, "sha256": h.hexdigest()}\n        ghi()\n        for mh in ("zero_shot", "ft"):\n            khoa = f"{nhanh}/{mh}"\n            if khoa in st["ket_qua"]:\n                continue\n            if time.time() - cfg["_t_bat_dau"] > het_giay:\n                raise DungHetGio(khoa)\n            goc = nap_goc(thu_mh)\n            model = PeftModel.from_pretrained(goc, str(duong_ad)) if mh == "ft" else goc\n            t0 = time.time()\n            du = sinh(model, tok, mau, thiet_bi, tran_theo="nguon")\n            giay = (time.time() - t0) / max(1, len(mau))\n            (thu_ra / f"sinh_{nhanh}_{mh}.jsonl").write_text("\\n".join(json.dumps(r, ensure_ascii=False) for r in du) + "\\n", encoding="utf-8")\n            tm, th = D.cham(pd.DataFrame(mau), pd.DataFrame(du), nhanh=nhanh, tap_da_thay=tap)\n            pid = {m["id"]: m["product_id"] for m in mau}\n            cot = [c for c in tm.columns if not c.startswith("_")]\n            (thu_ra / f"sinh_diem_{nhanh}_{mh}.jsonl").write_text(\n                "\\n".join(json.dumps(D._sach({**{c: r[c] for c in cot}, "product_id": pid[r["id"]]}), ensure_ascii=False) for r in tm.to_dict("records")) + "\\n", encoding="utf-8")\n            st["ket_qua"][khoa] = D._sach({"so_mau": len(mau), "giay_moi_mau": round(giay, 3), "sinh_lap": int(tm.sinh_lap.sum()), "thieu_hyp": th["thieu_hyp"],\n                                           "ti_le_cat_loi_dan": float(tm.da_cat_loi_dan.mean()), "chrf": th["diem_chinh"], "theo_huong": th["theo_huong"],\n                                           "theo_nhom": th["theo_nhom"], "cat_loi_dan": th["cat_loi_dan"], "dong_da_thay_moi": th.get("dong_da_thay_moi")})\n            ghi()\n            print(f"{khoa} XONG chrF {th[\'diem_chinh\'][\'chrf_macro_huong\']:.2f} so mau {len(mau)} giay/mau {giay:.3f}", flush=True)\n            del model, goc\n            gc.collect()\n            torch.cuda.empty_cache()\n    st["xong"] = all(f"{n}/{m}" in st["ket_qua"] for n in cfg["nhanh"] for m in ("zero_shot", "ft"))\n    ghi()\n    bc["dat"] = st["xong"]\n    print("DO_TEST XONG" if st["xong"] else "DO_TEST CHUA XONG", flush=True)\n\n\ndef thu_b5(cfg, bc):\n    import torch\n    import transformers\n    from peft import LoraConfig, PeftModel, get_peft_model\n    from transformers import AutoTokenizer\n    import peft\n\n    torch.manual_seed(cfg["seed"])\n    random.seed(cfg["seed"])\n    thiet_bi = "cuda:0"\n    bc["moi_truong"] = {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,\n                        "gpu": torch.cuda.get_device_name(0), "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),\n                        "cuda_khả_dụng": torch.cuda.is_available()}\n    print(bc["moi_truong"], flush=True)\n    thu_mh = tim_mot("REVISION.txt").parent\n    bc["mo_hinh"] = thu_mh.joinpath("REVISION.txt").read_text().split()\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n\n    goc_mau = doc_mau(cfg["nhanh"])\n    bc["so_mau_pool"] = len(goc_mau)\n    ma, bo = xay_mau(goc_mau, tok, cfg["max_len"])\n    bc["so_mau_vuot_max_len_bi_bo"] = bo\n    bc["so_token_toan_bo_1_epoch"] = sum(len(m["ids"]) for m in ma)\n    v0 = ma[0]\n    print("MAU MA HOA:\\n", tok.decode(v0["ids"])[:600], "\\nPHAN TINH LOSS:\\n", tok.decode([x for x in v0["labels"] if x != -100])[:300], flush=True)\n    train = [m for m in ma if m["fold5"] != cfg["fold_kiem"]]\n    kiem_all = [m for m in ma if m["fold5"] == cfg["fold_kiem"]]\n    bc["so_mau_train"], bc["so_mau_kiem"] = len(train), len(kiem_all)\n    if set(m["id"] for m in train) & set(m["id"] for m in kiem_all):\n        raise RuntimeError("giao_train_kiem")\n    rng = random.Random(cfg["seed"])\n    kiem = rng.sample(kiem_all, cfg["so_mau_kiem"])\n\n    t = time.time()\n    model = nap_goc(thu_mh)\n    bc["giay_nap_mo_hinh"] = round(time.time() - t, 1)\n    model = get_peft_model(model, LoraConfig(r=cfg["rank"], lora_alpha=2 * cfg["rank"], lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"))\n    # Đo 05/11/2026: không checkpoint + lô 4096 token nhanh nhất (5.873 token/giây, 52 GB); 8192 token không checkpoint tràn 95 GB\n    if cfg.get("gc", False):\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        model.enable_input_require_grads()\n    model.config.use_cache = False\n    tham_so = [p for p in model.parameters() if p.requires_grad]\n    bc["so_tham_so_lora"] = sum(p.numel() for p in tham_so)\n    print("tham so LoRA:", bc["so_tham_so_lora"], flush=True)\n\n    bc["loss_kiem_truoc"] = loss_kiem(model, tok, kiem, cfg["micro_bs"] * 2, thiet_bi)\n    print("loss kiem truoc:", bc["loss_kiem_truoc"], flush=True)\n\n    opt = torch.optim.AdamW(tham_so, lr=cfg["lr"], weight_decay=0.0)\n    ngan_sach = cfg.get("ngan_sach_token", 4096)\n    lo = chia_lo(train, cfg["micro_bs"], cfg["seed"], ngan_sach)\n    bc["ngan_sach_token"] = ngan_sach\n    co_batch = cfg["micro_bs"] * cfg["tich_luy"]        # batch hiệu dụng tính theo số mẫu\n    tong_buoc = cfg["so_buoc"]\n    model.train()\n    nhat_ky, tok_dem, tok_do, t_do = [], 0, 0, None\n    torch.cuda.reset_peak_memory_stats()\n    t0 = time.time()\n    it = iter(lo)\n    for buoc in range(tong_buoc):\n        if buoc == 5:                    # bỏ 5 bước đầu khỏi phép đo tốc độ\n            t_do, tok_do = time.time(), 0\n        tong, da_co = 0.0, 0\n        while da_co < co_batch:          # gom đủ batch hiệu dụng; lô dài có ít mẫu hơn nên số micro-batch thay đổi\n            k = next(it)\n            ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n            out = model(input_ids=ids, attention_mask=mask, labels=lab)\n            (out.loss * len(k) / co_batch).backward()\n            tong += out.loss.item() * len(k) / co_batch\n            da_co += len(k)\n            n = int(mask.sum().item())\n            tok_dem += n\n            tok_do += n\n        for g in opt.param_groups:\n            g["lr"] = cfg["lr"] * lich_lr(buoc, tong_buoc)\n        torch.nn.utils.clip_grad_norm_(tham_so, 1.0)\n        opt.step()\n        opt.zero_grad(set_to_none=True)\n        if buoc % 10 == 0 or buoc == tong_buoc - 1:\n            nhat_ky.append({"buoc": buoc, "loss": round(tong, 4), "giay": round(time.time() - t0, 1)})\n            print(nhat_ky[-1], flush=True)\n    bc["nhat_ky_loss"] = nhat_ky\n    bc["token_moi_giay"] = round(tok_do / max(time.time() - t_do, 1e-6), 1)\n    bc["tong_token_da_chay"] = tok_dem\n    bc["vram_dinh_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)\n    print("token/giay:", bc["token_moi_giay"], "VRAM dinh GB:", bc["vram_dinh_gb"], flush=True)\n\n    bc["loss_kiem_sau"] = loss_kiem(model, tok, kiem, cfg["micro_bs"] * 2, thiet_bi)\n    print("loss kiem sau:", bc["loss_kiem_sau"], flush=True)\n    bc["loss_giam"] = bc["loss_kiem_sau"] < bc["loss_kiem_truoc"]\n\n    ck = RA / "ck_b5"\n    model.save_pretrained(str(ck))\n    bc["ck_mb"] = round(sum(p.stat().st_size for p in ck.rglob("*") if p.is_file()) / 2**20, 1)\n    del model, opt\n    torch.cuda.empty_cache()\n    model = PeftModel.from_pretrained(nap_goc(thu_mh), str(ck))\n    bc["loss_kiem_sau_nap_lai"] = loss_kiem(model, tok, kiem, cfg["micro_bs"] * 2, thiet_bi)\n    bc["nap_lai_khop"] = abs(bc["loss_kiem_sau_nap_lai"] - bc["loss_kiem_sau"]) < 1e-3\n    print("loss sau nap lai:", bc["loss_kiem_sau_nap_lai"], "khop:", bc["nap_lai_khop"], flush=True)\n\n    # Sinh thử trên fold kiểm (không dùng test): chia đều theo hướng x loại\n    chon, n_moi = [], max(1, cfg["so_mau_sinh"] // 4)\n    id_co = {m["id"] for m in kiem}\n    for h in ("zh->vi", "vi->zh"):\n        for l in ("title", "mo_ta"):\n            ung = [m for m in goc_mau if m["fold5"] == cfg["fold_kiem"] and m["huong"] == h and m["loai"] == l and m["phan"] == 0 and m["id"] in id_co]\n            chon += rng.sample(ung, min(n_moi, len(ung)))\n    t = time.time()\n    du_ft = sinh(model, tok, chon, thiet_bi)\n    bc["giay_sinh_moi_mau_ft"] = round((time.time() - t) / max(1, len(chon)), 2)\n    t = time.time()\n    with model.disable_adapter():\n        du_zs = sinh(model, tok, chon, thiet_bi)\n    bc["giay_sinh_moi_mau_zs"] = round((time.time() - t) / max(1, len(chon)), 2)\n\n    import pandas as pd\n    import cham_diem as D\n    tc = pd.DataFrame(chon)\n    for ten, du in (("fine_tune", du_ft), ("zero_shot", du_zs)):\n        (RA / f"sinh_{ten}.jsonl").write_text("\\n".join(json.dumps(r, ensure_ascii=False) for r in du) + "\\n", encoding="utf-8")\n        tm, th = D.cham(tc, pd.DataFrame(du), nhanh=cfg["nhanh"])\n        bc[f"cham_{ten}"] = th.get("diem_chinh")\n        bc[f"cham_{ten}_ti_le_cat_loi_dan"] = float(tm.da_cat_loi_dan.mean())\n    bc["vi_du_sinh"] = [{"id": m["id"], "huong": m["huong"], "loai": m["loai"], "nguon": m["nguon"][:200], "dich": m["dich"][:200],\n                         "ft": a["hyp_tho"][:200], "zs": b["hyp_tho"][:300]} for m, a, b in zip(chon, du_ft, du_zs)]\n\n\ndef do_toc_do(cfg, bc):\n    """B5: so token/giây và VRAM giữa các cách chia lô (kể cả lô dài nhất để biết có tràn không). Chỉ đo tốc độ, không lưu mô hình."""\n    import torch\n    from peft import LoraConfig, get_peft_model\n    from transformers import AutoTokenizer\n\n    thiet_bi = "cuda:0"\n    thu_mh = tim_mot("REVISION.txt").parent\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    ma, _ = xay_mau(doc_mau(cfg["nhanh"]), tok, cfg["max_len"])\n    train = [m for m in ma if m["fold5"] != cfg["fold_kiem"]]\n    model = get_peft_model(nap_goc(thu_mh), LoraConfig(r=cfg["rank"], lora_alpha=2 * cfg["rank"], lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"))\n    model.enable_input_require_grads()\n    model.config.use_cache = False\n    model.train()\n    tham_so = [p for p in model.parameters() if p.requires_grad]\n    opt = torch.optim.AdamW(tham_so, lr=cfg["lr"], weight_decay=0.0)\n    dai_nhat = sorted(train, key=lambda m: -len(m["ids"]))\n    bc["do_toc_do"] = []\n    for c in cfg["cac_cau_hinh"]:\n        kq = dict(c)\n        if c["gc"]:\n            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        else:\n            model.gradient_checkpointing_disable()\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n        try:\n            xau = cat_ngan_sach(dai_nhat[:64], c["micro_bs"], c["ngan_sach"])[0]     # lô tệ nhất\n            ids, lab, mask = lo_tensor(xau, tok, thiet_bi)\n            model(input_ids=ids, attention_mask=mask, labels=lab).loss.backward()\n            opt.zero_grad(set_to_none=True)\n            kq["vram_lo_dai_nhat_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 1)\n            it = iter(chia_lo(train, c["micro_bs"], cfg["seed"], c["ngan_sach"]))\n            dem, t0 = 0, None\n            for buoc in range(cfg["so_buoc_do"]):\n                if buoc == 3:\n                    dem, t0 = 0, time.time()\n                da_co = 0\n                while da_co < 32:\n                    k = next(it)\n                    ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n                    (model(input_ids=ids, attention_mask=mask, labels=lab).loss * len(k) / 32).backward()\n                    da_co += len(k)\n                    dem += int(mask.sum().item())\n                opt.step()\n                opt.zero_grad(set_to_none=True)\n            kq["token_moi_giay"] = round(dem / (time.time() - t0), 1)\n            kq["vram_dinh_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 1)\n        except torch.OutOfMemoryError:\n            kq["tran_bo_nho"] = True\n            opt.zero_grad(set_to_none=True)\n            torch.cuda.empty_cache()\n        bc["do_toc_do"].append(kq)\n        print(kq, flush=True)\n    bc["dat"] = True\n\n\ndef do_nguyen_nhan(cfg, bc):\n    """Tìm chỗ chậm trong một bước huấn luyện: thời gian từng phần trên cùng một bộ lô cố định, GPU có chạy hết công suất không, và vài biến thể."""\n    import os\n    import threading\n    import torch\n    import torch.nn.functional as F\n    from peft import LoraConfig, get_peft_model\n    from transformers import AutoTokenizer\n\n    thiet_bi = "cuda:0"\n    thu_mh = tim_mot("REVISION.txt").parent\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    ma, _ = xay_mau(doc_mau(cfg["nhanh"]), tok, cfg["max_len"])\n    train = [m for m in ma if m["fold5"] != cfg["fold_kiem"]]\n    lo = chia_lo(train, cfg["micro_bs"], cfg["seed"], cfg["ngan_sach_token"])[:cfg["so_lo_do"]]\n    goc = nap_goc(thu_mh)\n    c = goc.config\n    tong_tok = sum(len(m["ids"]) for k in lo for m in k)\n    o_dem = sum(len(k) * max(len(m["ids"]) for m in k) for k in lo)\n    tinh_loss = sum(sum(x != -100 for x in m["labels"]) for k in lo for m in k)\n    bc["mo_hinh_cau_truc"] = {"so_lop": c.num_hidden_layers, "hidden": c.hidden_size, "tu_vung": c.vocab_size, "attn": getattr(c, "_attn_implementation", None),\n                              "so_cpu": os.cpu_count(), "dtype": str(next(goc.parameters()).dtype)}\n    bc["lo_do"] = {"so_lo": len(lo), "token_that": tong_tok, "o_tinh_gom_dem": o_dem, "ti_le_dem": round(1 - tong_tok / o_dem, 3),\n                   "token_tinh_loss": tinh_loss, "ti_le_token_tinh_loss": round(tinh_loss / tong_tok, 3)}\n    print(bc["mo_hinh_cau_truc"], bc["lo_do"], flush=True)\n\n    def dong_ho():\n        torch.cuda.synchronize()\n        return time.perf_counter()\n\n    def forward(model, ids, lab, mask, chi_dich):\n        if not chi_dich:\n            return model(input_ids=ids, attention_mask=mask, labels=lab).loss\n        trong = model.base_model.model                       # chỉ tính lm_head ở vị trí có loss, tránh ma trận logits toàn bộ\n        h = trong.model(input_ids=ids, attention_mask=mask).last_hidden_state[:, :-1]\n        nhan = lab[:, 1:]\n        chon = nhan != -100\n        return F.cross_entropy(trong.lm_head(h[chon]).float(), nhan[chon])\n\n    def do_bien_the(model, ten, chi_dich=False, lay_gpu=False):\n        tham_so = [p for p in model.parameters() if p.requires_grad]\n        opt = torch.optim.AdamW(tham_so, lr=1e-4, weight_decay=0.0)\n        model.train()\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n        tg = {"tensor": 0.0, "forward": 0.0, "backward": 0.0, "optimizer": 0.0}\n        mau_gpu, dung = [], threading.Event()\n\n        def lay():\n            while not dung.is_set():\n                r = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,clocks.sm,clocks.max.sm,power.draw,power.limit,temperature.gpu", "--format=csv,noheader,nounits"], capture_output=True, text=True)\n                mau_gpu.append(r.stdout.strip())\n                time.sleep(1)\n\n        if lay_gpu:\n            threading.Thread(target=lay, daemon=True).start()\n        dem = 0\n        for i, k in enumerate(lo):\n            t = dong_ho()\n            ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n            t1 = dong_ho()\n            loss = forward(model, ids, lab, mask, chi_dich)\n            t2 = dong_ho()\n            loss.backward()\n            t3 = dong_ho()\n            torch.nn.utils.clip_grad_norm_(tham_so, 1.0)\n            opt.step()\n            opt.zero_grad(set_to_none=True)\n            t4 = dong_ho()\n            if i >= 3:                    # bỏ vài lô đầu\n                tg["tensor"] += t1 - t\n                tg["forward"] += t2 - t1\n                tg["backward"] += t3 - t2\n                tg["optimizer"] += t4 - t3\n                dem += int(mask.sum().item())\n        dung.set()\n        tong = sum(tg.values())\n        kq = {"ten": ten, "token_moi_giay": round(dem / tong, 1), "vram_dinh_gb": round(torch.cuda.max_memory_allocated() / 2**30, 1),\n              "ti_trong_%": {a: round(100 * b / tong, 1) for a, b in tg.items()}, "so_tham_so_lora": sum(p.numel() for p in tham_so)}\n        if mau_gpu:\n            kq["nvidia_smi_mau"] = mau_gpu[len(mau_gpu) // 2:len(mau_gpu) // 2 + 4]\n        bc["bien_the"].append(kq)\n        print(kq, flush=True)\n        del opt\n\n    # Mốc: chỉ chạy suy luận mô hình gốc, không LoRA, không gradient\n    goc.eval()\n    t = dong_ho()\n    dem = 0\n    with torch.no_grad():\n        for k in lo:\n            ids, lab, mask = lo_tensor(k, tok, thiet_bi)\n            goc(input_ids=ids, attention_mask=mask)\n            dem += int(mask.sum().item())\n    bc["suy_luan_goc_token_moi_giay"] = round(dem / (dong_ho() - t), 1)\n    print("suy luan goc token/giay:", bc["suy_luan_goc_token_moi_giay"], flush=True)\n    bc["bien_the"] = []\n\n    def lora(model, r=16, dropout=0.05, muc_tieu="all-linear", **kw):\n        return get_peft_model(model, LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=dropout, target_modules=muc_tieu, task_type="CAUSAL_LM"), **kw)\n\n    for ten, args, chi_dich, lay_gpu in [\n            ("hien_tai_all_linear_r16", {}, False, True),\n            ("khong_dropout", {"dropout": 0.0}, False, False),\n            ("chi_attention_qkvo", {"muc_tieu": ["q_proj", "k_proj", "v_proj", "o_proj"]}, False, False),\n            ("r64", {"r": 64}, False, False),\n            ("lm_head_chi_vi_tri_dich", {}, True, False),\n            ("lora_cung_bf16", {"autocast_adapter_dtype": False}, False, False)]:\n        m = lora(goc, **args)\n        try:\n            do_bien_the(m, ten, chi_dich, lay_gpu)\n        except Exception:\n            bc["bien_the"].append({"ten": ten, "loi": traceback.format_exc()[-500:]})\n            print(bc["bien_the"][-1], flush=True)\n        goc = m.unload()\n        del m\n    bc["dat"] = True\n\n\ndef main(cfg):\n    import os\n    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")\n    bc = {"cau_hinh": cfg, "dat": False}\n    t = time.time()\n    try:\n        if cfg["che_do"] == "do_test" and cfg.get("nguon_do") == "test":\n            kiem_khoa_test_do()\n            bc["khoa_test"] = "chi_co_do_khoa_dung_ten"          # lần đo test thật: Dataset test riêng được gắn; kiem_ro_ri_test kiểm không rò vào pool\n        else:\n            kiem_khoa_test()\n            bc["khoa_test"] = "khong_co_file_test_trong_input"\n        cai_thu_vien()\n        if cfg["che_do"] == "do_test":\n            cfg["_t_bat_dau"] = t\n            try:\n                chay_do_test(cfg, bc)\n            except DungHetGio as e:\n                bc["dung_het_gio"] = str(e)\n                print("KET THUC PHIEN CO TRAT TU, dung o", e, flush=True)\n        elif cfg["che_do"] in ("b6", "b78"):\n            cfg["_t_bat_dau"] = t\n            try:\n                (chay_b6 if cfg["che_do"] == "b6" else chay_b78)(cfg, bc)\n            except DungHetGio as e:\n                bc["dung_het_gio"] = str(e)            # kết thúc phiên bình thường để Kaggle lưu đầu ra; phiên sau chạy tiếp\n                print("KET THUC PHIEN CO TRAT TU, dung o", e, flush=True)\n        elif cfg["che_do"] == "do_toc_do":\n            do_toc_do(cfg, bc)\n        elif cfg["che_do"] == "do_nguyen_nhan":\n            do_nguyen_nhan(cfg, bc)\n        elif cfg["che_do"] == "thu_b5":\n            thu_b5(cfg, bc)\n            bc["dat"] = bool(bc.get("loss_giam") and bc.get("nap_lai_khop"))\n        else:\n            raise RuntimeError(f"che_do_chua_ho_tro:{cfg[\'che_do\']}")\n    except Exception:\n        bc["loi"] = traceback.format_exc()\n        print(bc["loi"], flush=True)\n    bc["tong_giay"] = round(time.time() - t, 1)\n    ten_bc = f"bao_cao_{cfg.get(\'ten_trang_thai\', cfg[\'che_do\'])}.json"\n    (RA / ten_bc).write_text(json.dumps(bc, ensure_ascii=False, indent=2, default=str), encoding="utf-8")\n    print("KET QUA:", json.dumps({k: v for k, v in bc.items() if k not in ("vi_du_sinh", "cau_hinh", "nhat_ky_loss", "b6", "b78")}, ensure_ascii=False, default=str)[:3000], flush=True)\n    if "loi" in bc:\n        raise RuntimeError(f"loi, xem {ten_bc}")\n', encoding='utf-8')
Path('kaggle_quota.py').write_text('# -*- coding: utf-8 -*-\n"""Quota Survey A tren Kaggle - ban sua loi 10/11/2026.\n\nBa loi da sua so voi ban screen cu:\n1. DO DAI: loc theo DO DAI THAT (dung tokenizer + dung khuon chat) TRUOC khi\n   lay mau, roi huan luyen dung tap do. Khong con chuyen "ke hoach cat 960 /\n   luc chay bo mau": assert so mau bi bo = 0.\n2. NGAN SACH: moi buoc dung ~8.000 token DICH; di qua danh sach dung mot lan,\n   khong cycle. So buoc = tong token dich / 8.000 (tinh tu du lieu).\n3. LOSS: chuan hoa theo token dich cua ca buoc (tong loss / tong token dich),\n   khong chia theo so mau.\n\ns_g tinh tren token DICH. Ngach tinh theo token dich. Tong ngan sach tinh theo\ncompute (~4M token ca prompt) de so sanh cong bang; tong token dich tuong ung\nduoc tinh tu du lieu.\n\nKhong doc test. Khong sua du lieu.\n"""\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport kaggle_train as K\n\nVAO = Path("/kaggle/input")\nRA = Path("/kaggle/working")\n\nCHI_DAN_TITLE = "Translate this product title from {SRC} to {TGT}. Output only the translation.\\n\\n{NGUON}"\nCHI_DAN_CUM = (\n    "Translate this product\'s attribute list from {SRC} to {TGT}. The attributes are separated by \\"; \\" "\n    "and each is written as \\"name: value\\". Keep the same number and order of attributes, separated by \\"; \\", "\n    "and output only the translation.\\nProduct title (context only, do not translate): {TITLE}\\n\\n{NGUON}"\n)\nCHI_DAN_ATTR = "Translate this attribute from {SRC} to {TGT}. Product title (context only, do not translate): {TITLE}\\n\\n{NGUON}"\n\nTEN_NGU = {"zh->vi": ("Chinese", "Vietnamese"), "vi->zh": ("Vietnamese", "Chinese")}\nHUONG = {"1688": "zh->vi", "tiki": "vi->zh"}\n\n# Khuôn lời dẫn kiểu Alpaca cho EcomGPT (theo cấu hình của Quang: có mục "### Input"); chi_dan = phần chỉ dẫn, nguon = văn bản nguồn.\nALPACA = ("Below is an instruction that describes a task, paired with an input that provides further context. "\n          "Write a response that appropriately completes the request.\\n\\n### Instruction:\\n{chi_dan}\\n\\n### Input:\\n{nguon}\\n\\n### Response:")\n\n\ndef tao_khuon(tok, kieu="qwen"):\n    """Trả {"lam": hàm mẫu -> chuỗi lời dẫn, "duoi": chuỗi nối sau bản dịch khi huấn luyện}.\n\n    qwen: khuôn chat Qwen3 (tắt suy luận), bản dịch kết thúc bằng <|im_end|> và xuống dòng, đúng như mọi lần chạy từ trước.\n    ecomgpt: khuôn Alpaca, bản dịch kết thúc bằng </s> (eos của tokenizer BLOOM)."""\n    if kieu == "qwen":\n        return {"lam": lambda m: tok.apply_chat_template(m["messages"][:1], tokenize=False,\n                                                           add_generation_prompt=True, enable_thinking=False),\n                "duoi": K.NHAC_EOS + "\\n"}\n    if kieu == "ecomgpt":\n        if not tok.eos_token:\n            raise RuntimeError("TOKENIZER_KHONG_CO_EOS")\n        return {"lam": lambda m: ALPACA.format(chi_dan=m["chi_dan"], nguon=m["nguon"]), "duoi": tok.eos_token}\n    raise RuntimeError(f"KHUON_LA:{kieu}")\n\nCONFIGS = {\n    "cum": {\n        "natural": None,\n        "balanced": {"1688_title": 25, "1688_description": 25, "tiki_title": 25, "tiki_description": 25},\n        "title_heavy": {"1688_title": 40, "1688_description": 10, "tiki_title": 40, "tiki_description": 10},\n        "desc_heavy": {"1688_title": 2, "1688_description": 55, "tiki_title": 3, "tiki_description": 40},\n        "tiki_heavy": {"1688_title": 10, "1688_description": 15, "tiki_title": 35, "tiki_description": 40},\n    },\n    "pair": {\n        "natural": None,\n        "balanced": {"1688_title": 25, "1688_attribute": 25, "tiki_title": 25, "tiki_attribute": 25},\n        "title_heavy": {"1688_title": 40, "1688_attribute": 20, "tiki_title": 25, "tiki_attribute": 15},\n        "attr_heavy": {"1688_title": 10, "1688_attribute": 60, "tiki_title": 5, "tiki_attribute": 25},\n        "tiki_heavy": {"1688_title": 10, "1688_attribute": 20, "tiki_title": 30, "tiki_attribute": 40},\n    },\n}\nTHU_TU = {\n    "cum": ["1688_title", "1688_description", "tiki_title", "tiki_description"],\n    "pair": ["1688_title", "1688_attribute", "tiki_title", "tiki_attribute"],\n}\n\n\ndef tim_file(ten):\n    ds = [p for p in VAO.rglob(ten) if p.is_file()]\n    if not ds:\n        raise RuntimeError(f"khong_thay:{ten}")\n    return ds[0]\n\n\ndef doc_ref(tap, flt=0):\n    return pd.read_csv(tim_file(f"train_ref_{tap}_flt{flt}.csv"), encoding="utf-8-sig")\n\n\ndef doc_eval(tap):\n    return pd.read_csv(tim_file(f"eval_{tap}_sample.csv"), encoding="utf-8-sig")\n\n\ndef doc_pool(tap):\n    return K.doc_mau(tap)\n\n\ndef map_pool(recs):\n    return {(str(r["product_id"]), r["loai"]): r for r in recs}\n\n\ndef build_mau(ref, idx, pool_by, tap):\n    """Dung messages tu train_ref/eval_ref + pool. Tra list mau cho xay_mau."""\n    ra = []\n    tach = {}\n    idx = list(idx)\n    for dem, j in enumerate(idx, 1):\n        if dem % 50000 == 0:\n            print(f"  build_mau {tap}: {dem}/{len(idx)}", flush=True)\n        r = ref.iloc[int(j)]\n        pid = str(r["product_id"])\n        nhom = r["group"]\n        loai_ref = r["loai"]  # title | description | attribute\n        src = HUONG[nhom.split("_")[0]]\n        s, t = TEN_NGU[src]\n        title_rec = pool_by.get((pid, "title"))\n        rec = pool_by.get((pid, "title" if loai_ref == "title" else "mo_ta"))\n        if rec is None or title_rec is None:\n            continue\n        if loai_ref == "title":\n            nguon, dich = rec["nguon"], rec["dich"]\n            user = CHI_DAN_TITLE.format(SRC=s, TGT=t, NGUON=nguon)\n            chi_dan = CHI_DAN_TITLE.format(SRC=s, TGT=t, NGUON="").rstrip()\n            loai = "title"\n        elif loai_ref == "description":\n            nguon, dich = rec["nguon"], rec["dich"]\n            user = CHI_DAN_CUM.format(SRC=s, TGT=t, TITLE=title_rec["nguon"], NGUON=nguon)\n            chi_dan = CHI_DAN_CUM.format(SRC=s, TGT=t, TITLE=title_rec["nguon"], NGUON="").rstrip()\n            loai = "mo_ta"\n        else:  # attribute line\n            if pid not in tach:\n                tach[pid] = (str(rec["nguon"]).split("\\n"), str(rec["dich"]).split("\\n"))\n            ls, ld = tach[pid]\n            dong = int(r["dong"])\n            if dong < 0 or dong >= len(ls) or dong >= len(ld):\n                continue\n            nguon, dich = ls[dong], ld[dong]\n            user = CHI_DAN_ATTR.format(SRC=s, TGT=t, TITLE=title_rec["nguon"], NGUON=nguon)\n            chi_dan = CHI_DAN_ATTR.format(SRC=s, TGT=t, TITLE=title_rec["nguon"], NGUON="").rstrip()\n            loai = "mo_ta"\n        ra.append({\n            "chi_dan": chi_dan,\n            "id": f"{pid}_{loai_ref}_{r[\'dong\']}_{int(j)}",\n            "product_id": pid,\n            "huong": src,\n            "loai": loai,\n            "group": nhom,\n            "category": rec.get("category", "?"),\n            "fold5": int(r["fold5"]),\n            "nguon": nguon,\n            "dich": dich,\n            "nhanh": "cum" if tap == "cum" else "pair",\n            "e": float(r["e"]),\n            "tgt": float(r["tgt"]),\n            "messages": [{"role": "user", "content": user}, {"role": "assistant", "content": dich}],\n        })\n    return ra\n\n\ndef do_dai(mau, tok, lo=1000, khuon=None):\n    """Dien n_p (prompt), n_t (dich), n_chat = n_p + n_t, tokenize theo lo cho nhanh. khuon=None: khuon Qwen (mac dinh, dung de LAY MAU)."""\n    khuon = khuon or tao_khuon(tok, "qwen")\n    for i in range(0, len(mau), lo):\n        if i and (i // lo) % 50 == 0:\n            print(f"  do_dai: {i}/{len(mau)}", flush=True)\n        k = mau[i:i + lo]\n        ps = [khuon["lam"](m) for m in k]\n        as_ = [m["messages"][1]["content"] + khuon["duoi"] for m in k]\n        ip = tok(ps, add_special_tokens=False)["input_ids"]\n        ia = tok(as_, add_special_tokens=False)["input_ids"]\n        for m, a, b in zip(k, ip, ia):\n            m["n_p"], m["n_t"] = len(a), len(b)\n            m["n_chat"] = len(a) + len(b)\n\n\ndef lay_nhom_t(rng, pos, nt, T):\n    """Boc deu co hoan lai trong nhom cho toi khi du T token dich (vuot o mau cuoi)."""\n    if T <= 0 or len(pos) == 0:\n        return np.array([], dtype=np.int64)\n    mean = float(nt[pos].mean())\n    out, tot = [], 0.0\n    while tot < T:\n        k = max(10, int((T - tot) / mean * 1.1) + 10)\n        pick = pos[rng.choice(len(pos), size=k, replace=True)]\n        out.append(pick)\n        tot += float(nt[pick].sum())\n    out = np.concatenate(out)\n    cs = np.cumsum(nt[out])\n    cut = int(np.searchsorted(cs, T, side="left")) + 1\n    return out[:cut]\n\n\ndef ngach(mau, tap, cap, ngan_sach_compute, seed=42):\n    """Tra dict config -> {idx, s_g}; ngach tinh tren token DICH."""\n    giu = np.array([i for i, m in enumerate(mau) if m["n_chat"] <= cap], dtype=np.int64)\n    if len(giu) == 0:\n        raise RuntimeError(f"KHONG_CON_MAU:{tap}:cap{cap}")\n    grp = np.array([mau[i]["group"] for i in giu])\n    nt = np.zeros(len(mau), dtype=np.float64)      # chi so theo vi tri goc trong mau\n    nc = np.zeros(len(mau), dtype=np.float64)\n    for i in giu:\n        nt[i] = mau[i]["n_t"]\n        nc[i] = mau[i]["n_chat"]\n    r = float(nc[giu].sum() / max(nt[giu].sum(), 1.0))          # compute tren moi token dich\n    ngan_sach_t = int(ngan_sach_compute / max(r, 1e-9))\n    M = {g: float(nt[giu][grp == g].sum()) for g in THU_TU[tap] if (grp == g).any()}\n    tong = sum(M.values())\n    natural = {g: M[g] / tong for g in M}\n\n    ra = {}\n    for cname, cfg in CONFIGS[tap].items():\n        s = natural if cfg is None else {g: cfg[g] / 100.0 for g in THU_TU[tap]}\n        if abs(sum(s.values()) - 1) > 1e-9:\n            raise RuntimeError(f"S_G_KHONG_TONG_1:{tap}:{cname}")\n        rng = np.random.default_rng(seed)\n        parts = []\n        for g in THU_TU[tap]:\n            pos = giu[grp == g]\n            if len(pos) == 0:\n                if s.get(g, 0) > 0:\n                    raise RuntimeError(f"THIEU_NHOM:{tap}:{cname}:{g}")\n                continue\n            parts.append(lay_nhom_t(rng, pos, nt, s[g] * ngan_sach_t))\n        sel = np.concatenate([p for p in parts if len(p)])\n        rng.shuffle(sel)\n        ra[cname] = {"idx": sel, "s_g": s}\n\n    return ra, {\n        "tap": tap, "cap": cap, "n_tat_ca": len(mau), "n_giu": int(len(giu)), "n_bo": int(len(mau) - len(giu)),\n        "dai_nhat_tat_ca": int(max(m["n_chat"] for m in mau)),\n        "dai_nhat_giu": int(max(m["n_chat"] for m in (mau[i] for i in giu))),\n        "compute_tren_token_dich": round(r, 4),\n        "ngan_sach_token_dich": int(ngan_sach_t), "natural": natural,\n        "token_dich_toan_bo": int(nt[giu].sum()),\n    }\n\n\ndef gom_buoc_dich(lo, budget):\n    """Gom micro-batch thanh buoc sao cho moi buoc ~budget token DICH."""\n    buoc, cur, tot = [], [], 0\n    for k in lo:\n        t = sum(m["n_tgt"] for m in k)\n        if cur and tot + t > budget:\n            buoc.append(cur)\n            cur, tot = [], 0\n        cur.append(k)\n        tot += t\n        if tot >= budget:\n            buoc.append(cur)\n            cur, tot = [], 0\n    if cur:\n        buoc.append(cur)\n    return buoc\n\n\ndef chuan_bi(tap, tok, cfg):\n    cap = int(cfg["max_len"])\n    ref = doc_ref(tap, int(cfg.get("flt", 0)))\n    pool_by = map_pool(doc_pool(tap))\n    mau = build_mau(ref, range(len(ref)), pool_by, tap)\n    print(f"[{tap}] dung {len(mau)} mau, dang tokenize do dai that...", flush=True)\n    do_dai(mau, tok)\n    _, kiem = ngach(mau, tap, cap, float(cfg["ngan_sach_compute"]), int(cfg["seed"]))\n    print(f"[{tap}] kiem: {json.dumps(kiem, ensure_ascii=False)}", flush=True)\n    if kiem["dai_nhat_giu"] > cap:\n        raise RuntimeError(f"VUOT_CAP_SAU_LOC:{tap}:{kiem[\'dai_nhat_giu\']}>{cap}")\n    return ref, mau, kiem\n\n\nW_TITLE_ZHVI = 0.10      # trọng số CỐ ĐỊNH cho title trong metric chính (zh->vi), theo quyết định đã chốt\n\n\ndef chi_so_chinh(tap, th, eval_mau=None):\n    """Metric chinh: chrF zh->vi voi trong so CO DINH 10% title / 90% mo ta.\n\n    Ban cu lay trong so bang tong token dich cua mau eval; voi pair, moi dong\n    attribute chi 2-3 token nen trong so title tut thanh ~67%, sai han so voi\n    ti le trien khai ~8-10%. Nay dung hang so 0,10.\n    """\n    nhom = th.get("theo_nhom") or {}\n    t = (nhom.get("zh->vi/title") or {}).get("chrf_corpus")\n    m = (nhom.get("zh->vi/mo_ta") or {}).get("chrf_corpus")\n    if t is None or m is None:\n        return None\n    return W_TITLE_ZHVI * float(t) + (1 - W_TITLE_ZHVI) * float(m)\n\n\ndef guardrail(th):\n    """Điểm riêng từng nhánh zh->vi để kiểm điều kiện không tụt quá 1 điểm."""\n    nhom = th.get("theo_nhom") or {}\n    return {k: (nhom[k]["chrf_corpus"] if k in nhom else None)\n            for k in ("zh->vi/title", "zh->vi/mo_ta")}\n\n\ndef ma_hoa(mau_sel, tok, cap, lo=500, khuon=None):\n    """Mã hoá theo lô, đúng khuôn chat như kaggle_train.xay_mau nhưng nhanh hơn.\n\n    Trả list {id, ids, labels, fold5, group, n_tgt}; lỗi nếu mẫu vượt cap. khuon=None: khuôn Qwen.\n    """\n    khuon = khuon or tao_khuon(tok, "qwen")\n    ra = []\n    for i in range(0, len(mau_sel), lo):\n        k = mau_sel[i:i + lo]\n        ps = [khuon["lam"](m) for m in k]\n        as_ = [m["messages"][1]["content"] + khuon["duoi"] for m in k]\n        ip = tok(ps, add_special_tokens=False)["input_ids"]\n        ia = tok(as_, add_special_tokens=False)["input_ids"]\n        for m, a, b in zip(k, ip, ia):\n            if len(a) + len(b) > cap:\n                raise RuntimeError(f"VUOT_CAP_KHI_MA_HOA:{m[\'id\']}:{len(a)+len(b)}>{cap}")\n            ra.append({"id": m["id"], "ids": a + b, "labels": [-100] * len(a) + b,\n                       "fold5": m["fold5"], "group": m["group"], "n_tgt": len(b)})\n    return ra\n\n\ndef do_dem(ma, tap):\n    """Ti le token dich thuc te theo nhom."""\n    nhom = {g: 0 for g in THU_TU[tap]}\n    for m in ma:\n        nhom[m["group"]] = nhom.get(m["group"], 0) + m["n_tgt"]\n    tong = sum(nhom.values()) or 1\n    return {g: nhom.get(g, 0) / tong for g in THU_TU[tap]}, tong\n\n\ndef cham_mot(model, tok, eval_mau, tap, thiet_bi, zs_cache, khuon=None):\n    import cham_diem as D\n    lam = khuon["lam"] if khuon else None\n    du = K.sinh(model, tok, eval_mau, thiet_bi, bs=32, ngan_sach=32768, lam_prompt=lam)\n    if tap not in zs_cache:      # zero-shot giống nhau cho mọi cấu hình cùng bộ -> chỉ sinh 1 lần\n        with model.disable_adapter():\n            zs_cache[tap] = K.sinh(model, tok, eval_mau, thiet_bi, bs=32, ngan_sach=32768, lam_prompt=lam)\n    tm, th = D.cham(pd.DataFrame(eval_mau), pd.DataFrame(du), nhanh=tap)\n    _, th_zs = D.cham(pd.DataFrame(eval_mau), pd.DataFrame(zs_cache[tap]), nhanh=tap)\n    return th, th_zs, du\n\n\ndef chay_job(job, cfg, tok, thu_mh, cache, ng, zs_cache):\n    import torch\n    from peft import LoraConfig, get_peft_model\n    tap, config = job["tap"], job["config"]\n    seed = int(job.get("seed", cfg["seed"]))\n    torch.manual_seed(seed)\n    random.seed(seed)\n    cap = int(cfg["max_len"])\n    mh = cfg.get("mo_hinh") or {}\n    kieu = mh.get("khuon", "qwen")\n    khuon = tao_khuon(tok, kieu)\n    ref, mau, kiem = cache[tap]\n    mau_sel = [mau[int(i)] for i in ng[config]["idx"]]\n    ma = ma_hoa(mau_sel, tok, cap, khuon=khuon)\n    ti_le, tong_dich = do_dem(ma, tap)\n    lech = {g: abs(ti_le.get(g, 0) - ng[config]["s_g"].get(g, 0)) for g in THU_TU[tap]}\n    # Danh sach mau lay theo token Qwen; voi mo hinh khac tokenizer, ti le token dich theo token cua chinh mo hinh do lech nhe so voi s_g\n    # nen chi bao lech (khong dung job); voi Qwen van dung neu lech > 0,01 nhu truoc.\n    if kieu == "qwen" and max(lech.values()) > 0.01:\n        raise RuntimeError(f"LECH_NGACH:{config}:{lech}")\n    print(f"[{tap}/{config}/s{seed}] khuon={kieu} n={len(ma)} token_dich={tong_dich} lech_max={max(lech.values()):.4f}", flush=True)\n\n    thiet_bi = "cuda:0"\n    torch.cuda.reset_peak_memory_stats()\n    goc = K.nap_goc(thu_mh, attn=mh.get("attn", "sdpa"))\n    if mh.get("gc"):\n        goc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        goc.enable_input_require_grads()\n    model = get_peft_model(goc, LoraConfig(r=cfg["rank"], lora_alpha=2 * cfg["rank"], lora_dropout=0.05,\n                                           target_modules="all-linear", task_type="CAUSAL_LM"))\n    model.config.use_cache = False\n    tham_so = [p for p in model.parameters() if p.requires_grad]\n    opt = torch.optim.AdamW(tham_so, lr=cfg["lr"], weight_decay=0.0)\n    lo = K.chia_lo(ma, cfg["micro_bs"], seed, cfg["ngan_sach_token"])\n    buoc = gom_buoc_dich(lo, int(cfg["so_token_moi_buoc"]))\n    model.train()\n    t0 = time.time()\n    tok_dem = 0\n    nhat_ky = []\n    for i, cac_lo in enumerate(buoc):\n        N = sum(m["n_tgt"] for k in cac_lo for m in k)\n        if N <= 0:\n            raise RuntimeError(f"BUOC_KHONG_CO_TOKEN:{i}")\n        tong = 0.0\n        for k in cac_lo:\n            n_k = sum(m["n_tgt"] for m in k)\n            ids, lab, mask = K.lo_tensor(k, tok, thiet_bi)\n            out = model(input_ids=ids, attention_mask=mask, labels=lab)\n            (out.loss * (n_k / N)).backward()\n            tong += out.loss.item() * (n_k / N)\n            tok_dem += int(mask.sum().item())\n        for g in opt.param_groups:\n            g["lr"] = cfg["lr"] * K.lich_lr(i, len(buoc))\n        torch.nn.utils.clip_grad_norm_(tham_so, 1.0)\n        opt.step()\n        opt.zero_grad(set_to_none=True)\n        if i % 50 == 0 or i == len(buoc) - 1:\n            nhat_ky.append({"buoc": i, "loss": round(tong, 4), "giay": round(time.time() - t0, 1)})\n            print(nhat_ky[-1], flush=True)\n        if cfg.get("dung_sau_buoc") and i + 1 >= int(cfg["dung_sau_buoc"]):     # chi cho chay khoi: dung som, vi du 8 buoc\n            nhat_ky.append({"buoc": i, "loss": round(tong, 4), "giay": round(time.time() - t0, 1), "dung_som": True})\n            print("DUNG SOM (chay khoi)", nhat_ky[-1], flush=True)\n            break\n    giay = time.time() - t0\n    bo_nho_gb = round(torch.cuda.max_memory_allocated() / 2 ** 30, 2)\n\n    eval_ref = doc_eval(tap)\n    pool_by = map_pool(doc_pool(tap))\n    eval_mau = build_mau(eval_ref, range(len(eval_ref)), pool_by, tap)\n    if cfg.get("so_eval") and int(cfg["so_eval"]) < len(eval_mau):\n        rng = random.Random(seed)\n        eval_mau = rng.sample(eval_mau, int(cfg["so_eval"]))\n    th, th_zs, du = cham_mot(model, tok, eval_mau, tap, thiet_bi, zs_cache, khuon)\n    n_eval = len(du)\n    out_dir = RA / "quota" / f"{tap}_{config}_s{seed}"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    model.save_pretrained(str(out_dir))\n    (out_dir / "sinh.jsonl").write_text(\n        "\\n".join(json.dumps(x, ensure_ascii=False) for x in du), encoding="utf-8")\n    del model, opt\n    torch.cuda.empty_cache()\n    return {\n        "tap": tap, "config": config, "seed": seed, "s_g": ng[config]["s_g"], "realized": ti_le,\n        "mo_hinh": mh.get("ten", "qwen3-4b"), "khuon": kieu, "attn": mh.get("attn", "sdpa"), "gc": bool(mh.get("gc")),\n        "bo_nho_dinh_gb": bo_nho_gb, "buoc_ke_hoach": len(buoc), "dung_som": bool(cfg.get("dung_sau_buoc")),\n        "lech_max": max(lech.values()), "n_mau": len(ma), "bo_max_len": 0,\n        "so_buoc": (nhat_ky[-1]["buoc"] + 1 if cfg.get("dung_sau_buoc") else len(buoc)), "token_dich": int(tong_dich),\n        "ngan_sach_token_moi_buoc": int(cfg["so_token_moi_buoc"]),\n        "ngan_sach_token": cfg["ngan_sach_token"], "micro_bs": cfg["micro_bs"],\n        "rank": cfg["rank"], "lr": cfg["lr"], "max_len": cap,\n        "token_compute": tok_dem, "giay": round(giay, 1), "token_moi_giay": round(tok_dem / max(giay, 1e-6), 1),\n        "loss": nhat_ky[-1]["loss"] if nhat_ky else None, "nhat_ky": nhat_ky,\n        "eval_n": n_eval, "chrf": th.get("diem_chinh"), "chrf_zs": th_zs.get("diem_chinh"),\n        "diem_chinh": chi_so_chinh(tap, th, eval_mau),\n        "guardrail_zhvi": guardrail(th),\n        "theo_huong": th.get("theo_huong"), "theo_huong_zs": th_zs.get("theo_huong"),\n        "theo_nhom": th.get("theo_nhom"), "theo_nhom_zs": th_zs.get("theo_nhom"),\n        "adapter": str(out_dir),\n    }\n\n\ndef do_dem_mau(mau_sel, tap):\n    """Ti le token dich theo nhom, tinh truc tiep tu n_t (khong can ma hoa)."""\n    nhom = {g: 0 for g in THU_TU[tap]}\n    for m in mau_sel:\n        nhom[m["group"]] = nhom.get(m["group"], 0) + m["n_t"]\n    tong = sum(nhom.values()) or 1\n    return {g: nhom.get(g, 0) / tong for g in THU_TU[tap]}, tong\n\n\ndef thong_ke_dai(mau_sel, tok, cap, khuon, tap, lo=500):\n    """Ma hoa TOAN BO mau da chon bang tokenizer + khuon cua chinh mo hinh (khong tao model): do dai, so mau vuot cap, ti le token dich theo nhom,\n    ti le mau ket thuc dung ma EOS, va vi du loi dan that de nguoi doc kiem bang mat."""\n    dai, nhom_t, eos_dung, n_eos = [], {g: 0 for g in THU_TU[tap]}, 0, 0\n    eos_id = tok.convert_tokens_to_ids(khuon["duoi"].strip()) if khuon["duoi"].strip() else None\n    vi_du = {}\n    for i in range(0, len(mau_sel), lo):\n        k = mau_sel[i:i + lo]\n        ps = [khuon["lam"](m) for m in k]\n        as_ = [m["messages"][1]["content"] + khuon["duoi"] for m in k]\n        ip = tok(ps, add_special_tokens=False)["input_ids"]\n        ia = tok(as_, add_special_tokens=False)["input_ids"]\n        for m, p, a, b in zip(k, ps, ip, ia):\n            dai.append(len(a) + len(b))\n            nhom_t[m["group"]] = nhom_t.get(m["group"], 0) + len(b)\n            if eos_id is not None and eos_id != tok.unk_token_id:\n                n_eos += 1\n                eos_dung += int(eos_id in b[-2:])\n            if m["loai"] not in vi_du:\n                vi_du[m["loai"]] = {"loi_dan": p[:700], "dich": m["messages"][1]["content"][:120], "n_p": len(a), "n_t": len(b)}\n    tong = sum(nhom_t.values()) or 1\n    return {"n": len(dai), "n_vuot_cap": int(sum(1 for x in dai if x > cap)), "dai_nhat": int(max(dai) if dai else 0),\n            "dai_tb": round(sum(dai) / max(len(dai), 1), 1), "token_dich": int(tong),\n            "ti_le_token_dich_theo_nhom": {g: round(nhom_t.get(g, 0) / tong, 4) for g in THU_TU[tap]},\n            "ti_le_mau_ket_thuc_dung_eos": (round(eos_dung / n_eos, 4) if n_eos else None), "vi_du": vi_du}\n\n\ndef kiem_job(job, cfg, tok, cache, ng):\n    """Dry run: kiem ngach + kiem mau hoa toan bo bang tokenizer/khuon cua mo hinh, khong tao model."""\n    tap, config = job["tap"], job["config"]\n    cap = int(cfg["max_len"])\n    mh = cfg.get("mo_hinh") or {}\n    kieu = mh.get("khuon", "qwen")\n    ref, mau, kiem = cache[tap]\n    sel = ng[config]["idx"]\n    mau_sel = [mau[int(i)] for i in sel]\n    ti_le, tong_dich = do_dem_mau(mau_sel, tap)\n    s = ng[config]["s_g"]\n    lech = {g: abs(ti_le.get(g, 0) - s.get(g, 0)) for g in THU_TU[tap]}\n\n    # ma hoa toan bo mau chon (khong raise) de biet truoc co mau nao vuot cap trong token cua mo hinh nay\n    tk = thong_ke_dai(mau_sel, tok, cap, tao_khuon(tok, kieu), tap)\n    dai_thu = tk["dai_nhat"]\n    bo_thu = tk["n_vuot_cap"]\n\n    compute = sum(m["n_p"] + m["n_t"] for m in mau_sel)\n    buoc_uoc = int(math.ceil(tong_dich / max(int(cfg["so_token_moi_buoc"]), 1)))\n    ra = {\n        "tap": tap, "config": config, "seed": int(job.get("seed", cfg["seed"])),\n        "s_g": s, "realized": ti_le,\n        "lech_max": max(lech.values()), "n_mau": len(mau_sel), "bo_max_len": int(bo_thu),\n        "so_buoc_uoc": buoc_uoc, "token_dich": int(tong_dich), "token_compute": int(compute),\n        "ngan_sach_token_dich": kiem["ngan_sach_token_dich"],\n        "ngan_sach_token_moi_buoc": int(cfg["so_token_moi_buoc"]),\n        "max_len": cap, "dai_nhat_mau_ma_hoa": int(dai_thu), "mo_hinh": mh.get("ten", "qwen3-4b"), "khuon": kieu,\n        "thong_ke_mo_hinh": tk,\n        "dai_nhat_mau_chon": max(m["n_chat"] for m in mau_sel),\n        "kiem": kiem,\n    }\n    if kieu == "qwen" and max(lech.values()) > 0.01:\n        ra["loi"] = f"LECH_NGACH:{lech}"\n    if bo_thu or dai_thu > cap:\n        ra["loi"] = f"MA_HOA_THU_LOI:bo={bo_thu}:dai={dai_thu}"\n    print(f"[dry {tap}/{config}] n={len(mau_sel)} dich={tong_dich} buoc~{buoc_uoc} "\n          f"compute={compute} lech_max={max(lech.values()):.4f} bo_thu={bo_thu} dai_thu={dai_thu}", flush=True)\n    return ra\n\n\ndef tim_thu_muc_mo_hinh(ten_thu_muc=None):\n    """Thu muc chua REVISION.txt cua mo hinh. ten_thu_muc=None: nhu truoc (REVISION.txt dau tien); co ten: chi lay thu muc ten do, vi co the gan nhieu mo hinh."""\n    if not ten_thu_muc:\n        return K.tim_mot("REVISION.txt").parent\n    ds = sorted(p for p in VAO.rglob(f"{ten_thu_muc}/REVISION.txt") if p.is_file())\n    if not ds:\n        raise RuntimeError(f"khong_thay_mo_hinh:{ten_thu_muc}")\n    return ds[0].parent\n\n\ndef main(cfg):\n    import torch\n    import transformers\n    from transformers import AutoTokenizer\n    K.cai_thu_vien()\n    torch.manual_seed(cfg["seed"])\n    random.seed(cfg["seed"])\n    dry = cfg.get("che_do") == "quota_dry"\n    mh = cfg.get("mo_hinh") or {}\n    bc = {"moi_truong": {"torch": torch.__version__, "transformers": transformers.__version__,\n                         "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"},\n          "che_do": cfg.get("che_do"), "mo_hinh": mh, "viec": []}\n    thu_mh = tim_thu_muc_mo_hinh(mh.get("ten_thu_muc"))\n    print("thu muc mo hinh:", thu_mh, flush=True)\n    tok = AutoTokenizer.from_pretrained(str(thu_mh))\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"\n    # Danh sach mau LAY theo do dai token cua Qwen (cung mot danh sach cho moi mo hinh); mo hinh khac chi doi tokenizer luc huan luyen\n    tok_mau = tok\n    if mh.get("tok_lay_mau_thu_muc"):\n        tok_mau = AutoTokenizer.from_pretrained(str(tim_thu_muc_mo_hinh(mh["tok_lay_mau_thu_muc"])))\n        print("tokenizer lay mau:", mh["tok_lay_mau_thu_muc"], flush=True)\n\n    cache = {}\n    zs_cache = {}\n    ng_cache = {}\n    for tap in sorted({j["tap"] for j in cfg["viec"]}):\n        cache[tap] = chuan_bi(tap, tok_mau, cfg)\n        bc.setdefault("kiem_tung_tap", {})[tap] = cache[tap][2]\n        (RA / ("quota_dry.json" if dry else "quota_report.json")).write_text(\n            json.dumps(bc, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n\n    for job in cfg["viec"]:\n        try:\n            tap, seed = job["tap"], int(job.get("seed", cfg["seed"]))\n            if (tap, seed) not in ng_cache:\n                ng_cache[(tap, seed)], _ = ngach(cache[tap][1], tap, int(cfg["max_len"]),\n                                                 float(cfg["ngan_sach_compute"]), seed)\n            ng = ng_cache[(tap, seed)]\n            bc["viec"].append(kiem_job(job, cfg, tok, cache, ng) if dry\n                              else chay_job(job, cfg, tok, thu_mh, cache, ng, zs_cache))\n        except Exception as e:\n            import traceback\n            bc["viec"].append({"tap": job["tap"], "config": job["config"], "seed": job.get("seed", cfg["seed"]),\n                               "loi": f"{type(e).__name__}: {e}",\n                               "traceback": traceback.format_exc()[-3000:]})\n            print("LOI", job, e, flush=True)\n        (RA / ("quota_dry.json" if dry else "quota_report.json")).write_text(\n            json.dumps(bc, ensure_ascii=False, indent=1, default=str), encoding="utf-8")\n    print("XONG", len(bc["viec"]), flush=True)\n    return bc\n\n\nif __name__ == "__main__":\n    raise SystemExit("Nhung vao notebook, goi main(CAU_HINH)")\n', encoding='utf-8')


In [ ]:
import sys
sys.path.insert(0, '.')
import kaggle_quota
kaggle_quota.main(CAU_HINH)
